**Restart the Jupyter kernel before running any new session.**

Kernel > Restart Kernel and Clear All Outputs


Could you kindly update this markdown sale from the notebook, please?

# The Creative Prism
## Phase 1 — Full Studio Workflow
**v3.1— Adaptive Discovery, Routing, Team Configuration**
```
Stage 0    Routing              ← Director classifies: STUDIO / DIRECT / OUT_OF_SCOPE
Stage 1    Adaptive Discovery   ← Director-controlled loop, 1-6 turns, extraction toolkit
Stage 2    Reframing            ← Director proposes reframing, PIL confirms
Stage 3    Creative Brief       ← JSON format, validation gate
Stage 3a   Team Configuration   ← Director tunes Creative Team traits for this problem (NEW)
Stage 3b   Open Questions       ← Director surfaces to PIL before ideation
Stage 4a   Creator Pass 1       ← 3 directions, banded Verbalized Sampling
Stage 4b   Critic Pass 1        ← token-efficient evaluation, RESEARCH_REQUEST
Stage 4c   Researcher           ← responds to requests + autonomous contribution
Stage 4d   Creator Pass 2       ← refines directions based on research
Stage 4e   Critic Pass 2        ← final evaluation + synthesis → Director
Stage 5    Candidate Directions
Stage 6    Director Review      ← JSON format, reads from latest ideation cycle
Stage 6a   Iteration Loop       ← re-runs full Creative Team if ITERATE
Stage 7    Presentation
Stage 8    User Reaction
Stage 8b   Depth Extraction
Stage 9    Final Synthesis
```
**v1.5 changes:**
- Stage 0: Director routing — classifies request before engaging studio
- Stage 1: Adaptive discovery loop — Director controls turn count (1-6), deploys extraction toolkit
- Stage 2: Director proposes reframing to PIL for confirmation before briefing
- Stage 3a: Director configures Creative Team personality traits for this specific problem
- DIRECTOR_MODEL separate from SESSION_MODEL — Sonnet for Director, Haiku for team
- All PIL interactions are live input — no simulated personas
- Full conversation history stored in Studio Brief Document

---
## Cell 1 — Load Phase 0 Foundation

In [ ]:
import sys
!{sys.executable} -m pip install openai

In [1]:
# ── CELL 1 — LOAD FOUNDATION (OpenAI variant) ─────────────────────────────────
# Imports from engine_openai.py instead of engine.py.
# Everything else in the notebook is identical — all stage logic is
# model-agnostic. Only Cell 1 and Cell 2 differ from the Claude version.

import re
import json

from engine_openai import (
    client,
    create_blackboard,
    scribe_log,
    build_prompt,
    call_role,
    save_session,
    verify_engine,
    create_brief_doc,
    update_brief_doc,
    read_brief_doc,
    load_traits_matrix,
    weight_to_band,
    build_trait_profile,
    get_tunable_traits,
    validate_adjustments,
    validate_stage,
    generate_baseline,
    assemble_evaluation_payload,
    DEFAULT_MODEL,
)

engine_ready = verify_engine()
if not engine_ready:
    raise RuntimeError(
        "Engine verification failed.\n"
        "Run setup first:\n"
        "  mkdir prompts_openai && cp prompts/* prompts_openai/\n"
        "  mkdir sessions_openai\n"
        "  Add OPENAI_API_KEY to .env"
    )

traits_matrix = load_traits_matrix()
print(f"\nTraits matrix: {len(traits_matrix)} active traits")
print("OpenAI engine ready")

-- ENGINE VERIFICATION (OpenAI) ----------------------------
  API client initialized
  Key loaded: sk-proj-...hV4A
  Default model: gpt-5.4-mini

  OK  prompts_openai/SYSTEM_PROMPT.md <- constitutional layer
  OK  prompts_openai/director.md
  OK  prompts_openai/creator.md
  OK  prompts_openai/critic.md
  OK  prompts_openai/researcher.md
  OK  prompts_openai/scribe.md
  OK  prompts_openai/director_extraction_games.md

  OK  persona_traits_matrix_v2.csv (57 active traits)

  System foundation: ~1085 tokens
  All checks passed — OpenAI engine ready
------------------------------------------------------------

Traits matrix: 57 active traits
OpenAI engine ready


---
## Cell 2 — Session Configuration

All PIL interactions are live input.

In [2]:
# ── CELL 2 — SESSION CONFIGURATION (OpenAI variant) ───────────────────────────

DIRECTOR_MODEL = "gpt-5.4"
SESSION_MODEL = "gpt-5.4-mini"

# Session-level trait adjustments -- populated by Director in Stage 3a
session_adjustments = {"creator": {}, "critic": {}, "researcher": {}}

# Built trait profiles -- populated after Stage 3a
trait_profiles = {"creator": "", "critic": "", "researcher": ""}

print(f"Director model : {DIRECTOR_MODEL}")
print(f"Session model  : {SESSION_MODEL}")

Director model : gpt-5.4
Session model  : gpt-5.4-mini


---
## Cell 3 — Stage 0: Routing

The Director's first decision: is this a creative challenge for the studio,
a simple request to handle directly, or something out of scope entirely?

STUDIO → enter adaptive discovery.
DIRECT → Director responds in one turn, session ends.
OUT_OF_SCOPE → warm redirect, session ends.

In [3]:
# -- STAGE 0: ROUTING ------------------------------------------------------

# -- Scenario metadata (operator setup -- edit before running) ------------
# For experimental runs: set SCENARIO_ID and SCENARIO_CATEGORY before running.
# For live sessions: leave both as empty strings. PIL never sees this.
_scenario_id = ""  # e.g. "P-01"
_scenario_category = ""  # e.g. "stress_test"
if _scenario_id:
    print(f"  Scenario: {_scenario_id} [{_scenario_category}]")

print("=" * 60)
print("STAGE 0 -- ROUTING")
print("=" * 60)

initial_prompt = input(
    "Welcome to The Creative Prism. What would you like to explore today?\n> "
).strip()
# -- Baseline generation (Control B) -----------------------------------
# Zero-shot response to raw prompt. Stored for evaluation comparison.
# Uses SESSION_MODEL — same capability as Creative Team, architecture
# is the isolated variable. Called once, before any agents run.
baseline_response = generate_baseline(initial_prompt, model=SESSION_MODEL)
print("  Baseline generated and stored.")

# -- Create session --------------------------------------------------------
blackboard = create_blackboard(
    initial_prompt, system_version="2.1", director_model=DIRECTOR_MODEL
)
session_id = blackboard["session_metadata"]["session_id"]
brief_doc_path = create_brief_doc(session_id, initial_prompt)
scribe_log(
    blackboard, "SYSTEM", "session_start", f"Session initiated: '{initial_prompt}'"
)

# -- Write scenario metadata to blackboard (silent) -----------------------
if _scenario_id:
    blackboard["session_metadata"]["scenario_metadata"] = {
        "scenario_id": _scenario_id,
        "category": _scenario_category,
        "prompt_as_entered": initial_prompt,
    }

# -- Director routing call -------------------------------------------------
routing_task = (
    "A person has just arrived at The Creative Prism with this request:\n\n"
    f'"{initial_prompt}"\n\n'
    "Determine the right route for this request.\n\n"
    "STUDIO -- A creative challenge that benefits from structured exploration: "
    "brand development, strategic thinking, creative direction, problem reframing, "
    "experience design, naming, positioning, identity work, or any challenge "
    "where multiple perspectives would help. Most requests should route here.\n\n"
    "DIRECT -- A simple, factual, or straightforward request you can answer "
    "directly in one response. A quick opinion, a factual answer, a simple "
    "recommendation.\n\n"
    "OUT_OF_SCOPE -- Not what The Creative Prism is designed for: medical "
    "information, legal advice, homework help, code debugging, schedules, "
    "general knowledge queries, translation, or anything that is not a "
    "creative or strategic challenge.\n\n"
    "Return ONLY a JSON object -- no preamble, no markdown fences:\n\n"
    "{\n"
    '  "route": "STUDIO",\n'
    '  "message": "your response to the person"\n'
    "}\n\n"
    "If STUDIO: Warmly acknowledge what they have brought. Demonstrate that "
    "you understood it. Then ask ONE targeted opening question to begin "
    "understanding the person behind the request. The person may not be "
    "experienced with AI tools -- meet them where they are. Their initial "
    "prompt may be vague or incomplete and that is completely normal.\n\n"
    "If DIRECT: Your message is your complete, helpful response.\n\n"
    "If OUT_OF_SCOPE: Warmly explain what The Creative Prism is designed "
    "for -- creative exploration, brand development, strategic thinking, "
    "problem reframing -- and invite them to bring a creative challenge. "
    "Two to three sentences. Confident and redirecting, not apologetic."
)

routing_response = call_role(
    role="director",
    user_message=routing_task,
    context=initial_prompt,
    blackboard=blackboard,
    model=DIRECTOR_MODEL,
    max_tokens=400,
    brief_doc="",
)

# -- Parse routing response ------------------------------------------------
try:
    clean = routing_response.strip()
    if clean.startswith("```"):
        clean = clean.split("```")[1]
        if clean.startswith("json"):
            clean = clean[4:]
    routing = json.loads(clean.strip())
    session_route = routing.get("route", "STUDIO").upper()
    director_message = routing.get("message", "")
except (json.JSONDecodeError, ValueError):
    session_route = "STUDIO"
    director_message = routing_response.strip()

print(f"\nRoute: {session_route}\n")
print(f"Director: {director_message}")

# -- Handle non-studio routes ---------------------------------------------
if session_route == "OUT_OF_SCOPE":
    scribe_log(
        blackboard,
        "DIRECTOR",
        "routing_out_of_scope",
        f"Request routed OUT_OF_SCOPE: '{initial_prompt[:60]}'",
    )
    update_brief_doc(
        session_id, "DIRECTOR", "ROUTING", f"Route: OUT_OF_SCOPE\n\n{director_message}"
    )
    save_session(blackboard)
    print("\n" + "=" * 60)
    print("SESSION COMPLETE -- out of scope request handled gracefully.")
    print("No further cells need to be run.")
    print("=" * 60)

elif session_route == "DIRECT":
    scribe_log(
        blackboard,
        "DIRECTOR",
        "routing_direct",
        f"Request routed DIRECT: '{initial_prompt[:60]}'",
    )
    update_brief_doc(
        session_id, "DIRECTOR", "ROUTING", f"Route: DIRECT\n\n{director_message}"
    )
    save_session(blackboard)
    print("\n" + "=" * 60)
    print("SESSION COMPLETE -- direct response delivered.")
    print("No further cells need to be run.")
    print("=" * 60)

else:
    scribe_log(
        blackboard,
        "DIRECTOR",
        "routing_studio",
        "Request routed to STUDIO. Entering adaptive discovery.",
    )
    print(f"\nRouted to studio -- entering adaptive discovery")
    print(f"Session ID: {session_id[:8]}...")
    print(f"Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

STAGE 0 -- ROUTING
  Baseline generated and stored.

Route: STUDIO

Director: You’ve got more here than just a sandwich idea — you’ve got a dish people already ask for by name, a story tied to your grandmother Petrina, and a real willingness to work for it even without a big budget. That’s a strong place to start. Before we shape this into a real business, tell me this: when you imagine it working, do you picture a small local sandwich shop, selling at pop-ups and events, or making sandwiches from home for pickup and catering?

Routed to studio -- entering adaptive discovery
Session ID: 4d7a1f1f...
Reasoning trace: 3 entries


---
## Cell 4 — Stage 1: Adaptive Discovery

Director-controlled discovery loop. One to six turns.
The Director decides when it has enough signal — not the notebook.

Each turn: Director speaks → PIL responds → Director assesses signal.
The Director can deploy extraction techniques (sacrificial concepts,
forced choices, fill-in-the-blank, etc.) when signal is weak.

All PIL interactions are live input.

In [4]:
# -- STAGE 1: ADAPTIVE DISCOVERY -------------------------------------------

print("=" * 60)
print("STAGE 1 -- ADAPTIVE DISCOVERY")
print("=" * 60)

if session_route != "STUDIO":
    print("Skipped -- session was routed as", session_route)
else:

    MAX_DISCOVERY_TURNS = 6

    # -- Discovery state ---------------------------------------------------
    conversation_history = []
    all_assessments = []
    discovery_status = "CONTINUE"
    current_signals = []
    current_missing = []

    # The routing call already produced the Director's first message.
    conversation_history.append({"role": "director", "content": director_message})

    print(f"\n-- Turn 1 -------------------------------------------------")
    print(f"Director: {director_message}\n")

    # -- Collect first PIL response ----------------------------------------
    pil_response = input("> ").strip()
    conversation_history.append({"role": "pil", "content": pil_response})

    # -- Discovery loop ----------------------------------------------------
    turn = 1

    while discovery_status == "CONTINUE" and turn < MAX_DISCOVERY_TURNS:
        turn += 1

        # Format conversation for the Director
        history_formatted = "\n".join(
            [
                ("DIRECTOR: " if e["role"] == "director" else "PERSON: ") + e["content"]
                for e in conversation_history
            ]
        )

        # Ceiling pressure on final turn
        ceiling_note = ""
        if turn == MAX_DISCOVERY_TURNS:
            ceiling_note = (
                f"\n\nThis is turn {turn} of {MAX_DISCOVERY_TURNS} -- your last turn. "
                "Set status to READY and work with the signal you have."
            )

        discovery_task = (
            "You are the Director of The Creative Prism, in a discovery "
            "conversation with a person who has brought a creative challenge.\n\n"
            f"CONVERSATION SO FAR:\n{history_formatted}\n\n"
            f"TURN: {turn} of {MAX_DISCOVERY_TURNS}{ceiling_note}\n\n"
            "YOUR EXTRACTION TOOLKIT (use when signal is weak or vague):\n"
            "- Forced Choice: Does this feel more like A or B?\n"
            "- Elimination: Which of these feels least right?\n"
            "- Extremes Test: Push concept to an extreme, observe comfort\n"
            "- Reaction Snapshot: Quick gut reaction, before you think about it?\n"
            "- Fill in the Blank: This should feel more like ___ than ___\n"
            "- Analogy Probe: If this were a place/object/experience, what would it be?\n"
            "- Constraint Introduction: If you had to explain in one sentence, what matters most?\n"
            "- Signal Amplification: Reflect what you heard slightly amplified, invite correction\n"
            "- Micro-Ranking: Rank these 1-3 on instinct: X / Y / Z\n"
            "- Sacrificial Concept: Present a deliberately wrong/strange idea -- "
            "the value is in what the person pushes back against and why\n"
            "- Category Transplant: Propose the solution as if it belonged to "
            "a different industry entirely\n\n"
            "GUIDANCE:\n"
            "- After turn 2, if the person gives broad or vague answers, DEPLOY a specific\n"
            "  extraction technique. Do not ask another open-ended question. Use a forced\n"
            "  choice, fill-in-the-blank, sacrificial concept, or analogy probe instead.\n"
            "- Offer specific alternatives rather than asking how they see things.\n"
            "- The person may not be experienced with AI. Meet them where they are.\n"
            "- Vague or incomplete answers are normal -- your toolkit handles this.\n"
            "- Acknowledge what they said before your next question. Brief, genuine.\n"
            "- ONE question or probe per turn. Never stack multiple questions.\n"
            "- Use extraction techniques when standard questions produce vague signal.\n"
            "- You are trying to understand the PERSON, not just the problem.\n"
            "- Keep it conversational -- a smart, engaged collaborator.\n\n"
            "WHAT YOU NEED FOR A CREATIVE BRIEF:\n"
            "- The challenge: what they are actually trying to solve\n"
            "- The context: who it is for, where it lives, what the situation is\n"
            "- What success looks like: desired outcome\n"
            "- What to avoid: boundaries, anti-patterns, negative space\n"
            "- The person: what they value beneath what they have said\n\n"
            "CRITICAL RULE FOR STATUS:\n"
            "- If you set status to READY, your message must be a CLOSING STATEMENT,\n"
            "  not a question. Summarize what you have learned and signal that you are\n"
            "  about to move to the next phase. Do NOT ask a question and set READY.\n"
            "- If you still have a question to ask, set status to CONTINUE.\n\n"
            "RESPOND IN TWO PARTS:\n\n"
            "PART 1: Your message to the person. Conversational, warm, one "
            "question or probe. Acknowledge what they said first.\n\n"
            "---SIGNAL_ASSESSMENT---\n\n"
            "PART 2: A JSON signal assessment (the person will NOT see this):\n"
            "{\n"
            '  "signals_gathered": ["list each distinct signal established so far"],\n'
            '  "signals_missing": ["what you still need to understand"],\n'
            '  "technique_used": "none | forced_choice | elimination | extremes_test | '
            "reaction_snapshot | fill_blank | analogy_probe | constraint_intro | "
            'signal_amplification | micro_ranking | sacrificial_concept | category_transplant",\n'
            '  "status": "CONTINUE | READY",\n'
            '  "reason": "one sentence -- why you need more or why you have enough"\n'
            "}"
        )

        director_response = call_role(
            role="director",
            user_message=discovery_task,
            context=initial_prompt,
            blackboard=blackboard,
            model=DIRECTOR_MODEL,
            max_tokens=800,
            brief_doc=read_brief_doc(session_id),
        )

        # -- Parse the two-part response -----------------------------------
        if "---SIGNAL_ASSESSMENT---" in director_response:
            parts = director_response.split("---SIGNAL_ASSESSMENT---", 1)
            director_message = parts[0].strip()
            assessment_raw = parts[1].strip()
        else:
            director_message = director_response.strip()
            assessment_raw = json.dumps(
                {
                    "signals_gathered": current_signals,
                    "signals_missing": ["assessment delimiter not found"],
                    "technique_used": "none",
                    "status": "CONTINUE" if turn < MAX_DISCOVERY_TURNS else "READY",
                    "reason": "Signal assessment delimiter not found in response",
                }
            )

        # Parse signal assessment JSON
        try:
            clean_a = assessment_raw.strip()
            if clean_a.startswith("```"):
                clean_a = clean_a.split("```")[1]
                if clean_a.startswith("json"):
                    clean_a = clean_a[4:]
            # Find closing brace to handle trailing text
            brace_depth = 0
            json_end = 0
            for ci, ch in enumerate(clean_a):
                if ch == "{":
                    brace_depth += 1
                elif ch == "}":
                    brace_depth -= 1
                    if brace_depth == 0:
                        json_end = ci + 1
                        break
            if json_end > 0:
                clean_a = clean_a[:json_end]
            assessment = json.loads(clean_a.strip())
        except (json.JSONDecodeError, ValueError, IndexError):
            assessment = {
                "signals_gathered": current_signals,
                "signals_missing": ["assessment parsing failed"],
                "technique_used": "unknown",
                "status": "CONTINUE" if turn < MAX_DISCOVERY_TURNS else "READY",
                "reason": "JSON parsing failed",
            }

        discovery_status = assessment.get("status", "CONTINUE").upper()
        current_signals = assessment.get("signals_gathered", current_signals)
        current_missing = assessment.get("signals_missing", [])
        technique = assessment.get("technique_used", "none")
        reason = assessment.get("reason", "")
        all_assessments.append(assessment)

        conversation_history.append({"role": "director", "content": director_message})

        # -- Display -------------------------------------------------------
        print(f"-- Turn {turn} -------------------------------------------------")
        print(f"Director: {director_message}\n")
        if technique and technique != "none":
            print(f"  [technique: {technique}]")
        print(f"  [status: {discovery_status} | {reason}]")
        print()

        if discovery_status == "READY":
            # Safety net: if Director asked a question while setting READY,
            # collect the response before exiting
            if director_message.rstrip().endswith("?"):
                print("  [Director asked a question with READY — collecting response]")
                pil_response = input("> ").strip()
                conversation_history.append({"role": "pil", "content": pil_response})
            break

        # -- Collect PIL response ------------------------------------------
        pil_response = input("> ").strip()
        conversation_history.append({"role": "pil", "content": pil_response})

    # -- Store discovery in Blackboard and Brief Document ------------------
    blackboard["discovery"]["orientation_summary"] = initial_prompt
    blackboard["discovery"]["context_insights"] = current_signals
    blackboard["discovery"]["preferences"] = [
        s
        for s in current_signals
        if any(
            kw in s.lower()
            for kw in ["avoid", "prefer", "want", "don't", "hate", "love", "feel"]
        )
    ]

    # Check if sacrificial concept was used
    for idx, a in enumerate(all_assessments):
        if a.get("technique_used") == "sacrificial_concept":
            dir_idx = (idx + 1) * 2
            if dir_idx < len(conversation_history):
                probe = conversation_history[dir_idx].get("content", "")
                resp_idx = dir_idx + 1
                resp = (
                    conversation_history[resp_idx].get("content", "")
                    if resp_idx < len(conversation_history)
                    else ""
                )
                blackboard["discovery"]["sacrificial_exploration"][
                    "probe_prompt"
                ] = probe
                blackboard["discovery"]["sacrificial_exploration"][
                    "user_response"
                ] = resp
                blackboard["discovery"]["sacrificial_exploration"][
                    "insight_extracted"
                ] = f"Sacrificial concept deployed at turn {idx+2}. PIL reaction recorded."
            break

    # Record techniques used
    techniques_used = [
        a.get("technique_used", "none")
        for a in all_assessments
        if a.get("technique_used", "none") != "none"
    ]
    blackboard["discovery"]["exploratory_prompts"] = techniques_used

    # Store full conversation in Studio Brief Document
    discovery_log = "\n\n".join(
        [
            ("**Director:** " if e["role"] == "director" else "**Person:** ")
            + e["content"]
            for e in conversation_history
        ]
    )
    update_brief_doc(session_id, "DIRECTOR", "DISCOVERY CONVERSATION", discovery_log)

    if current_signals:
        signals_section = "\n".join(f"- {s}" for s in current_signals)
        update_brief_doc(session_id, "DIRECTOR", "SIGNALS GATHERED", signals_section)

    if current_missing:
        missing_section = "\n".join(f"- {m}" for m in current_missing)
        update_brief_doc(session_id, "DIRECTOR", "OPEN SIGNAL GAPS", missing_section)

    scribe_log(
        blackboard,
        "DIRECTOR",
        "discovery_complete",
        f"Adaptive discovery complete. {turn} turns. "
        f"{len(current_signals)} signals gathered. "
        f"Techniques: {techniques_used if techniques_used else 'none'}.",
    )

    print(f"\nDiscovery complete -- {turn} turns")
    print(f"Signals gathered: {len(current_signals)}")
    if techniques_used:
        print(f"Techniques used: {', '.join(techniques_used)}")
    if current_missing:
        print(f"Still missing: {', '.join(current_missing[:3])}")
    print(f"Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

STAGE 1 -- ADAPTIVE DISCOVERY

-- Turn 1 -------------------------------------------------
Director: You’ve got more here than just a sandwich idea — you’ve got a dish people already ask for by name, a story tied to your grandmother Petrina, and a real willingness to work for it even without a big budget. That’s a strong place to start. Before we shape this into a real business, tell me this: when you imagine it working, do you picture a small local sandwich shop, selling at pop-ups and events, or making sandwiches from home for pickup and catering?

-- Turn 2 -------------------------------------------------
Director: Got it — that gives this a real shape. Farmers markets first, then a food truck, tells me you’re not dreaming in the abstract; you’re looking for a path that can grow step by step without needing a huge upfront bet.

Quick instinct check: does this business feel more like **A) an old-family-recipe sandwich people line up for because it feels warm and homemade**, or **B) 

---
## Cell 5 — Stage 2 & 3: Reframing + Creative Brief

The Director proposes a reframing of the challenge to the PIL for confirmation.
This is the moment where the PIL sees whether the Director understood them.
After confirmation, the Director produces the structured creative brief.

The reframing step closes the discovery loop by
demonstrating understanding before the Creative Team is engaged.

In [5]:
# -- STAGE 2: REFRAMING ----------------------------------------------------
# -- STAGE 3: CREATIVE BRIEF -----------------------------------------------

print("=" * 60)
print("STAGE 2 -- REFRAMING")
print("STAGE 3 -- CREATIVE BRIEF")
print("=" * 60)

if session_route != "STUDIO":
    print("Skipped -- session was routed as", session_route)
else:

    # -- Build discovery summary from conversation -------------------------
    discovery_log_plain = "\n".join(
        [
            ("Director: " if e["role"] == "director" else "Person: ") + e["content"]
            for e in conversation_history
        ]
    )

    signals_summary = (
        "\n".join(f"- {s}" for s in current_signals)
        if current_signals
        else "None captured"
    )
    missing_summary = (
        "\n".join(f"- {m}" for m in current_missing) if current_missing else "None"
    )

    # -- Director proposes reframing ---------------------------------------
    reframe_task = (
        "You have just completed a discovery conversation. Now propose a "
        "reframing of the person's challenge.\n\n"
        f"DISCOVERY CONVERSATION:\n{discovery_log_plain}\n\n"
        f"SIGNALS GATHERED:\n{signals_summary}\n\n"
        f"GAPS REMAINING:\n{missing_summary}\n\n"
        "Your reframing should:\n"
        "- Show the person you understood not just what they said, but what they meant\n"
        "- Restate the challenge in a way that is sharper than how they put it\n"
        "- Surface any insight from the conversation they may not have seen themselves\n"
        "- Be 3-5 sentences -- clear, confident, specific to this person\n\n"
        "End by asking the person to confirm this is right, or to adjust anything "
        "that is off. This is the moment where they see whether you understood them."
    )

    reframe_response = call_role(
        role="director",
        user_message=reframe_task,
        context=initial_prompt,
        blackboard=blackboard,
        model=DIRECTOR_MODEL,
        max_tokens=400,
        brief_doc=read_brief_doc(session_id),
    )

    print("-- DIRECTOR REFRAMING ------------------------------------------")
    print(reframe_response)
    print()

    # -- PIL confirms or adjusts -------------------------------------------
    pil_confirmation = input("> ").strip()

    # -- Director reads PIL response -- branch on content ------------------
    signal_read_task = (
        f"The PIL has responded to your reframing:\n\n"
        f"YOUR REFRAMING:\n{reframe_response}\n\n"
        f"PIL RESPONSE:\n{pil_confirmation}\n\n"
        "Read this carefully. Determine which of these applies:\n\n"
        "A) CONFIRMED — the PIL agreed with the reframing, possibly with minor affirmation. "
        "Respond with one brief, natural acknowledgment (one sentence) and nothing else.\n\n"
        "B) NEW SIGNAL — the PIL added new information, corrected something, or shifted "
        "the framing. Respond by: (1) acknowledging what they added in one sentence, "
        "(2) stating how it changes your understanding of the challenge in one sentence. "
        "Do not re-present the full reframing. Just absorb and confirm the addition.\n\n"
        "C) SIGNIFICANT CORRECTION — the PIL said the reframing was substantially off. "
        "Respond by acknowledging the correction and asking ONE focused question to "
        "clarify what you missed. One sentence acknowledgment, one question.\n\n"
        "Keep your response to 2-3 sentences maximum. "
        "Do not repeat content from the reframing. Do not ask multiple questions."
    )

    signal_read_response = call_role(
        role="director",
        user_message=signal_read_task,
        context=initial_prompt,
        blackboard=blackboard,
        model=DIRECTOR_MODEL,
        max_tokens=150,
        brief_doc=read_brief_doc(session_id),
    )

    print()
    print(signal_read_response)
    print()

    # -- Store reframing exchange including Director acknowledgment --------
    update_brief_doc(
        session_id,
        "DIRECTOR",
        "REFRAMING",
        f"**Director:**\n{reframe_response}\n\n"
        f"**Person:**\n{pil_confirmation}\n\n"
        f"**Director acknowledged:**\n{signal_read_response}",
    )
    scribe_log(
        blackboard,
        "DIRECTOR",
        "reframing_confirmed",
        f"Reframing confirmed. Director acknowledged: {signal_read_response[:80]}",
    )

    blackboard["discovery"]["desired_outcome"] = reframe_response

    # -- Director synthesizes creative brief -------------------------------
    brief_task = (
        "Based on this confirmed understanding, produce a structured creative brief.\n\n"
        f"DISCOVERY CONVERSATION:\n{discovery_log_plain}\n\n"
        f"REFRAMING (confirmed by person):\n{reframe_response}\n\n"
        f"PERSON'S RESPONSE TO REFRAMING:\n{pil_confirmation}\n\n"
        f"DIRECTOR'S READING OF THAT RESPONSE:\n{signal_read_response}\n\n"
        f"SIGNALS GATHERED:\n{signals_summary}\n\n"
        "Return ONLY a JSON object -- no preamble, no markdown fences:\n\n"
        "{\n"
        '  "challenge": "one clear sentence -- the actual challenge, not the stated one",\n'
        '  "context": "2-3 sentences about situation, audience, and what matters",\n'
        '  "desired_result": "what success looks like -- specific to what was revealed",\n'
        '  "constraints": ["constraint 1", "constraint 2"],\n'
        '  "pressure_point": "one sentence naming the productive tension or paradox the Creative Team should press against -- not resolve, press against",\n'
        '  "assumptions": ["assumption the Director is making that the PIL should validate or correct", "assumption 2"]\n'
        "}\n\n"
        "Be specific. This brief becomes the Creative Team's working document. "
        "Include what the person explicitly said AND what their reactions revealed. "
        "The pressure_point is the creative instrument -- it names what makes this "
        "problem genuinely open rather than already solved. "
        "The assumptions are things the Director inferred that the PIL should "
        "confirm or correct -- not questions for the PIL to answer, but premises "
        "the Director is building on."
    )

    brief_response = call_role(
        role="director",
        user_message=brief_task,
        context=initial_prompt,
        blackboard=blackboard,
        model=DIRECTOR_MODEL,
        max_tokens=800,
        brief_doc=read_brief_doc(session_id),
    )

    # -- Parse brief -- JSON with validation gate --------------------------
    brief = blackboard["creative_brief"]

    try:
        clean = brief_response.strip()
        if clean.startswith("```"):
            clean = clean.split("```")[1]
            if clean.startswith("json"):
                clean = clean[4:]
        parsed = json.loads(clean.strip())

        brief["challenge"] = parsed.get("challenge", "")
        brief["context"] = parsed.get("context", "")
        brief["desired_result"] = parsed.get("desired_result", "")
        brief["constraints"] = parsed.get("constraints", [])
        brief["pressure_point"] = parsed.get("pressure_point", "")
        brief["open_questions"] = parsed.get(
            "assumptions", []
        )  # stored as open_questions for compatibility

        missing = [
            f
            for f in ["challenge", "context", "desired_result"]
            if not brief.get(f, "").strip()
        ]
        if missing:
            raise ValueError(f"Brief missing required fields: {missing}")

        print("Brief parsed successfully")

    except (json.JSONDecodeError, ValueError) as e:
        print(f"Brief parsing failed: {e}")
        print("Raw response:")
        print(brief_response)
        raise RuntimeError(
            "Brief parsing failed -- session cannot continue with an incomplete brief.\n"
            "Review the raw response above and re-run this cell."
        )

    # -- Store and display -------------------------------------------------
    update_brief_doc(
        session_id,
        "DIRECTOR",
        "CREATIVE BRIEF",
        f"**Challenge:** {brief['challenge']}\n\n"
        f"**Context:** {brief['context']}\n\n"
        f"**Desired result:** {brief['desired_result']}\n\n"
        f"**Constraints:** {', '.join(brief['constraints'])}\n\n"
        f"**Pressure point:** {brief.get('pressure_point', '')}\n\n"
        f"**Assumptions to validate:** {', '.join(brief['open_questions'])}",
    )

    scribe_log(
        blackboard,
        "DIRECTOR",
        "brief_synthesized",
        f"Creative brief produced. Challenge: '{brief['challenge'][:60]}...'",
    )
    validate_stage(blackboard, "brief")

    print()
    print(f"Challenge      : {brief['challenge']}")
    print(f"Context        : {brief['context'][:80]}...")
    print(f"Desired result : {brief['desired_result'][:80]}...")
    print(f"Constraints    : {brief['constraints']}")
    print(f"Pressure point : {brief.get('pressure_point', '')[:80]}")
    print(f"Assumptions    : {len(brief['open_questions'])}")
    print()
    print(f"Creative brief complete")
    print(f"Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

STAGE 2 -- REFRAMING
STAGE 3 -- CREATIVE BRIEF
-- DIRECTOR REFRAMING ------------------------------------------
What you’re really trying to build is not just a chicken parm business, but a **low-cost path to proving that one unforgettable sandwich can become a local habit**. The real challenge is to turn something people already love in informal settings into a farmers-market brand that feels **personal, homemade, and worth seeking out**, without drifting into “just another pizza-shop sandwich” or overinvesting before demand is proven. The hidden opportunity here is that **Petrina doesn’t have to carry the business through sentiment — the sandwich itself can earn attention first, and the brand story can deepen loyalty once people care**. So the core question is: how do you launch a small, believable first version that makes people talk, come back, and give you a real bridge from market stand to future food truck?

Does that feel right, or is there anything in that framing you’d want t

---
## Cell 5a — Stage 3a: Team Configuration

The Director configures the Creative Team's personality traits
for this specific problem. Adjusts 9 core trait weights and writes
a short team brief for each role (Creator, Critic, Researcher).

This is invisible to the PIL — it is the Director selecting the
right team for the job, exactly as a Creative Director would in
a real agency. The team brief is written to the Studio Brief Document
so every role reads it before acting.

In [6]:
# -- STAGE 3a: TEAM CONFIGURATION ------------------------------------------

print("=" * 60)
print("STAGE 3a -- TEAM CONFIGURATION")
print("=" * 60)

if session_route != "STUDIO":
    print("Skipped")
else:
    tunable_summary = ""
    for role_name in ["creator", "critic", "researcher"]:
        tunable = get_tunable_traits(role_name, traits_matrix)
        tunable_summary += "\n" + role_name.upper() + " tunable traits:\n"
        for t in tunable:
            tunable_summary += (
                "  "
                + t["trait"]
                + " (base: "
                + f"{t['base']:.2f}, range: {t['min']:.2f}-{t['max']:.2f})"
                f" -- {t['description']}\n"
            )

    config_task = (
        "You are the Director. The brief is complete. Configure the Creative "
        "Team for this specific problem.\n\n"
        f"BRIEF:\n"
        f"Challenge: {brief['challenge']}\n"
        f"Context: {brief['context']}\n"
        f"Constraints: {', '.join(brief['constraints'])}\n\n"
        "Your job is not to select traits that will help the team handle this "
        "problem competently. Your job is to configure a team that will push "
        "past the expected answer — toward directions the PIL would not have "
        "predicted.\n\n"
        "Think in combinations and tensions, not individual dials. A Creator "
        "with maximum imagination paired with maximum iconoclasm will challenge "
        "the premise of the brief. A Critic with maximum skepticism alongside "
        "maximum intellectual_generosity will destroy weak ideas while building "
        "on strong ones. Ask not what traits fit this domain — ask what "
        "perspective would genuinely surprise this problem.\n\n"
        "REQUIREMENTS:\n"
        "- Minimum 8 traits adjusted per role\n"
        "- At least 3 traits per role pushed to or near their maximum allowed value\n"
        "- Include traits from at least 3 different categories per role\n"
        "- Avoid defaulting to the obvious traits for the obvious domain\n\n"
        f"TUNABLE TRAITS (use verbatim names only):\n{tunable_summary}\n\n"
        "Return ONLY JSON -- no preamble:\n"
        "{\n"
        '  "creator_adjustments": {"trait_name": value, ...},\n'
        '  "critic_adjustments": {"trait_name": value, ...},\n'
        '  "researcher_adjustments": {"trait_name": value, ...},\n'
        '  "rationale": "2-3 sentences naming the creative intention, not just the problem type"\n'
        "}\n\n"
        "Only list traits you want to CHANGE from base. Unmentioned traits stay "
        "at their base values. Values must be within the allowed range. "
        "CRITICAL: trait names must match the list above exactly, character for character. "
        "Do not invent, abbreviate, or paraphrase trait names. "
        "Any name not found verbatim in the list will be discarded."
    )

    config_response = call_role(
        role="director",
        user_message=config_task,
        context=brief["challenge"],
        blackboard=blackboard,
        model=DIRECTOR_MODEL,
        max_tokens=900,
        brief_doc=read_brief_doc(session_id),
    )

    try:
        clean = config_response.strip()
        if clean.startswith("```"):
            clean = clean.split("```")[1]
            if clean.startswith("json"):
                clean = clean[4:]
        brace_depth = 0
        json_end = 0
        for ci, ch in enumerate(clean):
            if ch == "{":
                brace_depth += 1
            elif ch == "}":
                brace_depth -= 1
                if brace_depth == 0:
                    json_end = ci + 1
                    break
        if json_end > 0:
            clean = clean[:json_end]
        config = json.loads(clean.strip())

        for role_name in ["creator", "critic", "researcher"]:
            raw = config.get(role_name + "_adjustments", {})
            validated, warnings = validate_adjustments(role_name, raw, traits_matrix)
            session_adjustments[role_name] = validated
            for w in warnings:
                print(f"  Warning: {w}")

        for role_name in ["creator", "critic", "researcher"]:
            trait_profiles[role_name] = build_trait_profile(
                role_name, session_adjustments[role_name], traits_matrix
            )

        rationale = config.get("rationale", "")
        team_brief = "**Director reasoning:** " + rationale + "\n"
        for role_name in ["creator", "critic", "researcher"]:
            adj = session_adjustments[role_name]
            if adj:
                adj_str = ", ".join(f"{k}: {v:.2f}" for k, v in adj.items())
                team_brief += "\n**" + role_name.upper() + " adjustments:** " + adj_str

        update_brief_doc(session_id, "DIRECTOR", "TEAM CONFIGURATION", team_brief)
        scribe_log(
            blackboard,
            "DIRECTOR",
            "team_configured",
            f"Team configured. {rationale[:80]}...",
        )

        print(f"Team configured.\n\nRationale: {rationale}\n")
        for role_name in ["creator", "critic", "researcher"]:
            adj = session_adjustments[role_name]
            n = len(adj)
            print(
                f"{role_name.upper()}: {n} traits adjusted"
                + (f" -- {adj}" if adj else "")
            )

    except (json.JSONDecodeError, ValueError) as e:
        print(f"Team config parsing failed: {e}. Using default traits.")
        for role_name in ["creator", "critic", "researcher"]:
            trait_profiles[role_name] = build_trait_profile(
                role_name, {}, traits_matrix
            )

STAGE 3a -- TEAM CONFIGURATION
Team configured.

Rationale: I’m configuring a team to resist the obvious small-food-business answer of 'good sandwich, nice story, go sell it at markets' and instead search for the mechanism that can turn one item into a ritual people seek out. The Creator is pushed toward category-breaking imagination, the Critic toward hard-edged feasibility and disciplined pressure, and the Researcher toward pattern-finding across vending, brand formation, and staged growth so the work can discover a launch logic the founder would not have predicted but could actually build.

CREATOR: 20 traits adjusted -- {'imagination': 0.95, 'originality': 0.94, 'boldness': 0.89, 'iconoclasm': 0.84, 'inventiveness': 0.93, 'playfulness': 0.82, 'quirkiness': 0.76, 'reframing': 0.88, 'beginner_mind': 0.88, 'perspective_shifting': 0.84, 'experimentalism': 0.87, 'risk_taking': 0.84, 'improvisation': 0.84, 'pattern_recognition': 0.79, 'systems_thinking': 0.68, 'synthesis': 0.73, 'story_s

---
## Cell 4b — Stage 3b: Assumption Validation 

Director validates assumptions from the brief to the PIL before ideation begins.
Answers shape the brief and the Studio Brief Document.
PIL responds to open questions via live input.

In [ ]:
# ── STAGE 3b: ASSUMPTION VALIDATION ──────────────────────────
# v3.1 patch: assumptions now numbered (1. 2. 3.) so PIL can respond by number

print("═" * 60)
print("STAGE 3b — ASSUMPTION CHECK")
print("═" * 60)

assumptions = brief.get(
    "open_questions", []
)  # stored as open_questions for schema compatibility

if not assumptions:
    print("✓ No assumptions to validate — proceeding to ideation")
else:
    # -- Director presents assumptions for validation
    assumption_task = (
        f"The creative brief is complete. Before the Creative Team begins work, "
        f"surface the assumptions you made during briefing so the PIL can correct "
        f"any that are wrong.\n\n"
        f"ASSUMPTIONS TO VALIDATE:\n"
        + "\n".join(f"{i+1}. {a}" for i, a in enumerate(assumptions))
        + "\n\n"
        f"BRIEF CHALLENGE: {brief['challenge']}\n\n"
        f"Present these as a numbered list (1. 2. 3.) — keep the numbers from above. "
        f"These are premises you built on, things you inferred and want to confirm. "
        f"Also ask one single constraint question: "
        f"what would be completely off the table for this work?\n\n"
        f"Do NOT ask the PIL to answer strategic questions or do creative work. "
        f"You are asking them to correct wrong assumptions and name hard limits. "
        f"Keep it conversational and specific. One to three sentences per assumption. "
        f"Close with the constraint question."
    )

    assumption_response = call_role(
        role="director",
        user_message=assumption_task,
        context=brief["challenge"],
        blackboard=blackboard,
        model=DIRECTOR_MODEL,
        max_tokens=600,
        brief_doc=read_brief_doc(session_id),
    )

    print("── DIRECTOR ────────────────────────────────────────────────")
    print(assumption_response)
    print()

    pil_answers = input("> ").strip()

    # -- Director synthesizes PIL response into brief additions
    synthesis_task = (
        f"The PIL has responded to your assumption check:\n\n"
        f"YOUR ASSUMPTIONS:\n{assumption_response}\n\n"
        f"PIL RESPONSE:\n{pil_answers}\n\n"
        f"CURRENT BRIEF CHALLENGE: {brief['challenge']}\n\n"
        f"Do three things:\n"
        f"1. Acknowledge what the PIL said in one warm, specific sentence — "
        f"name what they corrected or confirmed, so they know they were heard.\n"
        f"2. State in plain language: what 1-3 things does this add or change in "
        f"the brief? Frame these as additions the Creative Team will now work with.\n"
        f"3. Confirm you are ready to begin.\n\n"
        f"Keep it to 3-4 sentences total. Do not ask follow-up questions. "
        f"The PIL should feel heard and should understand exactly what changed."
    )

    synthesis_response = call_role(
        role="director",
        user_message=synthesis_task,
        context=brief["challenge"],
        blackboard=blackboard,
        model=DIRECTOR_MODEL,
        max_tokens=200,
        brief_doc=read_brief_doc(session_id),
    )

    print(synthesis_response)
    print()

    # -- Store everything and enrich brief
    brief["open_questions"].append(f"PIL answered: {pil_answers}")
    brief["open_questions"].append(f"Director synthesis: {synthesis_response}")

    update_brief_doc(
        session_id,
        "DIRECTOR",
        "ASSUMPTION VALIDATION",
        f"**Director presented assumptions:**\n{assumption_response}\n\n"
        f"**PIL responded:**\n{pil_answers}\n\n"
        f"**Director synthesized and acknowledged:**\n{synthesis_response}",
    )
    scribe_log(
        blackboard,
        "DIRECTOR",
        "open_questions_answered",
        f"Assumption validation complete. Brief enriched. PIL acknowledged: {synthesis_response[:80]}",
    )

    print("✓ Assumptions validated — brief enriched")
    print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 3b — ASSUMPTION CHECK
════════════════════════════════════════════════════════════
── DIRECTOR ────────────────────────────────────────────────
Before I build on this, I want to make the assumptions visible so you can correct anything I got wrong.

1. I assumed your best first move is to become known for **one signature sandwich** rather than trying to launch with a bigger menu right away. The thinking here is that one standout item is cheaper to execute, easier to make consistently, and more likely to become the thing people remember. If that’s wrong, and you already know you need a broader offer from day one, say so.

2. I assumed the part people will remember most is **the taste itself** — especially the slight spice in the sauce and the quality of the ingredients — more than the family backstory on first encounter. In other words, I’m treating the story as support for the brand, not the main hook that gets people in

---
## Cell 5 — Stage 4a: Creator

The creative brief goes directly to the Creator first — not the Researcher.
The Creator generates candidate directions informed by the brief.
Verbalized Sampling applied: probability weights push toward lateral thinking.
The Researcher has not yet acted — the Creative Team works first.

In [8]:
# ── STAGE 4a: CREATOR — IDEATION PASS 1 ──────────────────────────────────────

print("═" * 60)
print("STAGE 4a — CREATOR: IDEATION (PASS 1)")
print("═" * 60)

creator_task = f"""You are working on this creative brief:

CHALLENGE: {brief['challenge']}
CONTEXT: {brief['context']}
DESIRED RESULT: {brief['desired_result']}
CONSTRAINTS: {", ".join(brief['constraints'])}

Generate 3 distinct conceptual directions using Verbalized Sampling.

━━━ VERBALIZED SAMPLING — ASSIGN THE BAND FIRST, THEN WRITE THE DIRECTION ━━━

One direction per band. Assign the band before writing. No two directions share a band.
The band determines the kind of thinking required — not a label applied afterward.

HIGH BAND (0.55–0.80)
The intelligent expected answer. A skilled practitioner working this brief carefully
would likely arrive here. Credible, grounded, implementable. Not obvious — but not
surprising. This is the direction that earns trust.

MID BAND (0.25–0.50)
A genuine reframe. The same problem understood differently. Requires stepping outside
the category's conventional assumptions. Not a variation on the high band direction —
a different way of understanding what the brief is actually asking for.

LOW BAND (0.08–0.22)
The direction that arrives from outside the expected territory entirely. Precise and
specific — lateral thinking is not loose thinking. This direction should make the room
go quiet. Uncomfortable and exact. A conventional approach would rarely reach here.

Assign your score within the band honestly. If it feels like 0.14, say 0.14.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

For each direction:
IDEA_ID: [A1, A2, A3]
BAND: [HIGH / MID / LOW]
PROBABILITY: [score within band]
TITLE: [short evocative name]
CONCEPT: [2-3 sentences — what is this direction?]
RATIONALE: [why this is worth exploring]
EMOTIONAL TERRITORY: [what feeling does this occupy?]
RESEARCH_REQUEST: [optional — one specific research question if external knowledge
would materially strengthen or ground this direction. Omit entirely if not needed.]

Do not write any preamble or analysis before your directions. Start directly with IDEA_ID: A1.

Be genuinely distinct — not three variations of the same idea."""

creator_response = call_role(
    role="creator",
    user_message=creator_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=SESSION_MODEL,
    max_tokens=1800,
    trait_profile=trait_profiles["creator"],
    brief_doc=read_brief_doc(session_id),
)

# Store as cycle 1
cycle_1 = {
    "cycle_number": 1,
    "creator_proposals": [{"raw_response": creator_response}],
    "critic_feedback": [],
    "synthesized_directions": [],
}
blackboard["ideation_cycles"].append(cycle_1)
update_brief_doc(session_id, "CREATOR", "DIRECTIONS — PASS 1", creator_response)
scribe_log(
    blackboard,
    "CREATOR",
    "ideation_complete",
    "Three directions generated using banded Verbalized Sampling. Cycle 1 logged.",
)
validate_stage(blackboard, "ideation")

print("── CREATOR DIRECTIONS ──────────────────────────────────────")
print(creator_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Creator pass 1 complete")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 4a — CREATOR: IDEATION (PASS 1)
════════════════════════════════════════════════════════════
── CREATOR DIRECTIONS ──────────────────────────────────────
IDEA_ID: A1  
BAND: HIGH  
PROBABILITY: 0.68  
TITLE: The One Sandwich Stand  
CONCEPT: Build the launch around a single, highly repeatable Chicken Parm sandwich sold at farmers markets with a tight, premium menu: the sandwich, maybe one optional side, and drinks. The operational idea is to make the product feel unmistakably homemade and special through consistency, a warm brand voice, and a visible “made for today” rhythm that rewards repeat visits.  
RATIONALE: This is the most credible launch path because it protects cash, reduces complexity, and gives the market one thing to remember. It turns the business into a focused habit-forming offer rather than a broad food concept, which fits both the founder’s capital constraints and the need to earn word-of-mouth through

---
## Cell 6 — Stage 4b: Critic

The Critic evaluates Creator directions and proposes a synthesis.
Reads the full Studio Brief Document including Creator output.
Constructive pressure — not rejection.

In [9]:
# ── STAGE 4b: CRITIC — EVALUATION PASS 1 ────────────────────────────────────

print("═" * 60)
print("STAGE 4b — CRITIC: EVALUATION (PASS 1)")
print("═" * 60)

creator_output = blackboard["ideation_cycles"][-1]["creator_proposals"][0][
    "raw_response"
]

critic_task = f"""The Creator has proposed three directions for the brief:

BRIEF CHALLENGE: {brief['challenge']}
DESIRED RESULT: {brief['desired_result']}

CREATOR\'S DIRECTIONS:
{creator_output}

Evaluate each direction using this tight format:

IDEA_ID: [matching the Creator\'s ID]
WHAT HOLDS: [1-2 sentences — what genuinely works and why]
WHAT DOESN\'T: [1-2 sentences — the specific weakness or risk]
REFINEMENT PATH: [one concrete action that would strengthen this direction]
RESEARCH_REQUEST: [optional — one specific research question if external knowledge
would materially help evaluate or strengthen this direction. Omit if not needed.]

After all three evaluations, note any cross-direction patterns if present:
PATTERN: [optional — if multiple directions share a common strength or weakness worth naming]
Present all  directions [A1, A2, A3] with their trade-offs stated factually -- do not rank or recommend.\n"

Be rigorous and efficient. Identify the sharpest refinement path for each direction."""

critic_response = call_role(
    role="critic",
    user_message=critic_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=SESSION_MODEL,
    max_tokens=2400,
    trait_profile=trait_profiles["critic"],
    brief_doc=read_brief_doc(session_id),
)

blackboard["ideation_cycles"][-1]["critic_feedback"].append(
    {"raw_response": critic_response}
)
update_brief_doc(session_id, "CRITIC", "EVALUATION — PASS 1", critic_response)
scribe_log(
    blackboard,
    "CRITIC",
    "evaluation_complete",
    "Pass 1 evaluation complete. Refinement paths identified.",
)
validate_stage(blackboard, "critic")

# Parse RESEARCH_REQUESTs from Creator and Critic for Researcher routing
creator_requests = re.findall(r"RESEARCH_REQUEST:\s*(.+?)(?=\n|$)", creator_output)
critic_requests = re.findall(r"RESEARCH_REQUEST:\s*(.+?)(?=\n|$)", critic_response)
research_requests = [
    f"[Creator]: {r.strip()}" for r in creator_requests if r.strip()
] + [f"[Critic]:  {r.strip()}" for r in critic_requests if r.strip()]

print("── CRITIC EVALUATION ───────────────────────────────────────")
print(critic_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Critic pass 1 complete")
if research_requests:
    print(f"  RESEARCH_REQUESTs found: {len(research_requests)}")
    for r in research_requests:
        print(f"    {r}")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 4b — CRITIC: EVALUATION (PASS 1)
════════════════════════════════════════════════════════════
── CRITIC EVALUATION ───────────────────────────────────────
IDEA_ID: A1  
WHAT HOLDS: This is the strongest feasibility-first launch frame: one product, one operating pattern, one quality standard. It directly addresses the capital constraint and makes repeatability, prep control, and customer recall much easier.  
WHAT DOESN'T: By itself, this is only a competent launch format; it does not yet create a distinct reason people seek out *this* sandwich versus any other good market sandwich. The mechanism of memorability is still mostly assumed rather than designed.  
REFINEMENT PATH: Add one signature differentiator that is visible and repeatable at the market level — for example, a named finishing move, a distinctive sauce application, or a consistent cut/packaging ritual that makes the sandwich instantly identifiable and talka

In [ ]:
print("ideation_cycles count:", len(blackboard["ideation_cycles"]))
last = blackboard["ideation_cycles"][-1]
print("Keys in last cycle:", list(last.keys()))
print("creator_proposals count:", len(last.get("creator_proposals", [])))
print("critic_feedback count:", len(last.get("critic_feedback", [])))

---
## Cell 7 — Stage 4c: Researcher

The Researcher acts AFTER the Creative Team — not before.
It reads the full Studio Brief Document including the ideation cycle
and provides contextual enrichment informed by what the team actually produced.
This is also when the bounded autonomous contribution trigger is evaluated:
the Researcher now has sufficient thinking on the Blackboard to identify genuine gaps.

In [10]:
# ── STAGE 4c: RESEARCHER ─────────────────────────────────────────────────────
print("═" * 60)
print("STAGE 4c — RESEARCHER")
print("═" * 60)
brief = blackboard["creative_brief"]
creator_output = blackboard["ideation_cycles"][-1]["creator_proposals"][0][
    "raw_response"
]
# Guard: catch missing Critic output before proceeding
critic_feedback_list = blackboard["ideation_cycles"][-1]["critic_feedback"]
if not critic_feedback_list:
    raise RuntimeError(
        "critic_feedback is empty — Critic output missing from blackboard. "
        "Re-run the Critic cell (Stage 4b) before proceeding."
    )
critic_output = critic_feedback_list[0]["raw_response"]

# Build request section if RESEARCH_REQUESTs were found
requests_section = ""
if research_requests:
    requests_section = (
        "\n\nSPECIFIC RESEARCH REQUESTS FROM THE CREATIVE TEAM:\n"
        + "\n".join(f"- {r}" for r in research_requests)
        + "\n\nAddress each of these specific requests directly before making any "
        "autonomous contribution."
    )

research_task = (
    f"You are supporting a creative studio session.\n\n"
    f"THE BRIEF:\n"
    f"Challenge: {brief['challenge']}\n"
    f"Context: {brief['context']}\n"
    f"Desired result: {brief['desired_result']}\n"
    f"{requests_section}\n\n"
    f"THE CREATIVE TEAM HAS PRODUCED:\n\n"
    f"CREATOR'S DIRECTIONS:\n{creator_output}\n\n"
    f"CRITIC'S EVALUATION:\n{critic_output}\n\n"
    f"Your task: source and cite specific external precedents, examples, or factual "
    f"findings that ground or expand what the Creative Team has produced.\n\n"
    f"For each finding:\n"
    f"TOPIC: [what area, domain, or precedent]\n"
    f"SOURCE: [name the specific source, study, institution, or example — be precise.\n"
    f"         Not 'research suggests' but the actual named source.]\n"
    f"FINDING: [what it shows]\n"
    f"RELEVANCE: [why this matters specifically for these directions]\n\n"
    f"Address specific requests first. Then make one autonomous contribution if you "
    f"have genuinely relevant external knowledge not already covered.\n\n"
    f"If the Creative Team's work is already well-grounded and no external knowledge "
    f"would add to it, say only:\n"
    f"RESEARCHER: No external contribution warranted for this cycle.\n\n"
    f"You are a sourcing agent. Do not evaluate the Creative Team's directions."
)

research_response = call_role(
    role="researcher",
    user_message=research_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=SESSION_MODEL,
    max_tokens=3000,
    trait_profile=trait_profiles["researcher"],
    brief_doc=read_brief_doc(session_id),
)

research_entry = {
    "research_id": f"R{len(blackboard['research_trace']) + 1:03d}",
    "initiated_by": "creative_team",
    "query": "Targeted requests + autonomous contribution post-ideation",
    "sources": ["researcher_knowledge"],
    "summary": research_response,
    "influence_on_ideation": "Delivered to Creative Team for Pass 2 refinement",
}
blackboard["research_trace"].append(research_entry)
brief["research_insights"].append(research_response)
update_brief_doc(session_id, "RESEARCHER", "RESEARCH FINDINGS", research_response)
scribe_log(
    blackboard,
    "RESEARCHER",
    "research_delivered",
    f"Research delivered. Entry R{len(blackboard['research_trace']):03d} logged.",
)

print("── RESEARCHER FINDINGS ─────────────────────────────────────")
print(research_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Research logged")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 4c — RESEARCHER
════════════════════════════════════════════════════════════
── RESEARCHER FINDINGS ─────────────────────────────────────
TOPIC: Farmers market vending rules for hot sandwich service  
SOURCE: U.S. Food and Drug Administration, *Model Food Code* (2022); local county/city health department food-vending permit rules; farmers market manager vendor guidelines  
FINDING: In the U.S., hot sandwich service at farmers markets is usually governed by a combination of temporary food establishment rules, commissary or approved kitchen requirements, handwashing access, temperature control standards for potentially hazardous foods, and permit/inspection requirements set by the local health authority. The FDA Model Food Code is the baseline reference many jurisdictions adapt, but the exact rules vary by state, county, and market operator.  
RELEVANCE: This directly affects whether the one-sandwich launch can be made-to

---
## Cell 7b — Stage 4d: Creator Refinement (Pass 2)

Creator reads the Critic's evaluation and the Researcher's findings.
Refines each of the three directions — does not replace them.
Incorporates research where relevant, addresses Critic's refinement paths.

In [11]:
# ── STAGE 4d: CREATOR — REFINEMENT PASS 2 ─────────────────────────────

print("═" * 60)
print("STAGE 4d — CREATOR: REFINEMENT (PASS 2)")
print("═" * 60)

creator_refinement_task = (
    f"You are the Creator. The Critic has evaluated your directions and the "
    f"Researcher has delivered findings. Now refine your work.\n\n"
    f"YOUR ORIGINAL DIRECTIONS:\n{creator_output}\n\n"
    f"CRITIC'S EVALUATION:\n{critic_output}\n\n"
    f"RESEARCHER'S FINDINGS:\n{research_response}\n\n"
    f"Refine each of your three directions. Do not replace them — refine them.\n"
    f"For each direction:\n"
    f"- Incorporate what the research adds, where it is genuinely relevant\n"
    f"- Address the Critic's refinement path\n"
    f"- Sharpen the concept based on what you now know\n\n"
    f"Use the same format as before:\n"
    f"IDEA_ID: [A1, A2, A3 — same IDs]\n"
    f"BAND: [same band — the territory doesn't change]\n"
    f"PROBABILITY: [may adjust within band if refinement shifted the direction]\n"
    f"TITLE: [may refine]\n"
    f"CONCEPT: [refined — 2-3 sentences]\n"
    f"RATIONALE: [updated to incorporate research and critique]\n"
    f"EMOTIONAL TERRITORY: [same or refined]\n\n"
    f"If a direction is already strong and the research adds nothing to it, "
    f"say so in one sentence and keep it as written. Refinement is not mandatory "
    f"where the direction already holds."
)

creator_refined_response = call_role(
    role="creator",
    user_message=creator_refinement_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=SESSION_MODEL,
    max_tokens=2400,  # raised from 1200 — 3-direction Pass 2 needs room
    trait_profile=trait_profiles["creator"],
    brief_doc=read_brief_doc(session_id),
)

# Store as cycle 2 (research-integrated refinement)
cycle_2 = {
    "cycle_number": 2,
    "creator_proposals": [{"raw_response": creator_refined_response}],
    "critic_feedback": [],
    "synthesized_directions": [],
}
blackboard["ideation_cycles"].append(cycle_2)
update_brief_doc(
    session_id, "CREATOR", "DIRECTIONS — PASS 2 (REFINED)", creator_refined_response
)
scribe_log(
    blackboard,
    "CREATOR",
    "refinement_complete",
    "Pass 2 refinement complete. Research and critique incorporated.",
)
validate_stage(blackboard, "ideation")

# Update creator_output to point to refined version for downstream cells
creator_output = creator_refined_response

print("── CREATOR REFINED DIRECTIONS ──────────────────────────────────")
print(creator_refined_response)
print("────────────────────────────────────────────────────────────────")
print(f"\n✓ Creator pass 2 complete")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 4d — CREATOR: REFINEMENT (PASS 2)
════════════════════════════════════════════════════════════
── CREATOR REFINED DIRECTIONS ──────────────────────────────────
IDEA_ID: A1  
BAND: HIGH  
PROBABILITY: 0.71  
TITLE: The One Sandwich Stand, Finished Clearly  
CONCEPT: Keep the launch tightly centered on one Chicken Parm sandwich, but add one unmistakable finishing ritual at the market — a signature sauce ladle, a precise cut, or a branded wrap/open-box presentation that makes the sandwich instantly identifiable. Structure the operation around what the research suggests is most practical: advance prep in an approved kitchen or commissary, with final heating, assembly, and service designed to fit temporary food rules and hot-holding requirements.  
RATIONALE: The core feasibility-first logic still holds, but now it is sharper: the low-cost market launch should be built to match local temporary food establishment rules, commi

---
## Cell 7c — Stage 4e: Critic Final Evaluation + Synthesis (Pass 2)

Critic evaluates the refined directions and produces one synthesis direction.
This is the Creative Team's final submission to the Director.
The synthesis direction may become a fourth candidate in the Director's review.

In [12]:
# ── STAGE 4e: CRITIC — FINAL EVALUATION + SYNTHESIS (PASS 2) ────────────────

print("═" * 60)
print("STAGE 4e — CRITIC: FINAL EVALUATION + SYNTHESIS (PASS 2)")
print("═" * 60)

critic_synthesis_task = (
    f"You are the Critic. The Creator has refined their three directions based on "
    f"your evaluation and the Researcher's findings.\n\n"
    f"BRIEF CHALLENGE: {brief['challenge']}\n"
    f"DESIRED RESULT: {brief['desired_result']}\n\n"
    f"REFINED DIRECTIONS:\n{creator_output}\n\n"
    f"This is the Creative Team's final submission before Director review.\n\n"
    f"For each refined direction:\n"
    f"IDEA_ID: [matching ID]\n"
    f"ASSESSMENT: [1-2 sentences — does the refinement hold? Is it stronger than before?]\n"
    f"REMAINING CONCERN: [one sentence if any weakness persists — or 'None']\n\n"
    f"Then produce ONE synthesis direction — something that combines the strongest "
    f"elements across the three refined directions into something none of them "
    f"reached alone:\n\n"
    f"SYNTHESIS_ID: S1\n"
    f"TITLE: [name]\n"
    f"CONCEPT: [2-3 sentences]\n"
    f"ORIGIN_IDEAS: [which direction IDs it draws from]\n"
    f"WHY THIS: [one sentence — what does this offer that the individual directions don't]\n\n"
    f"Present all four directions (A1, A2, A3, S1) with their trade-offs stated "
    f"factually -- do not rank or recommend. Close with:\n"
    f"The Creative Team submits these four directions to the Director for review."
)

critic_synthesis_response = call_role(
    role="critic",
    user_message=critic_synthesis_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=SESSION_MODEL,
    max_tokens=2400,
    trait_profile=trait_profiles["critic"],
    brief_doc=read_brief_doc(session_id),
)

blackboard["ideation_cycles"][-1]["critic_feedback"].append(
    {"raw_response": critic_synthesis_response}
)
update_brief_doc(
    session_id,
    "CRITIC",
    "FINAL EVALUATION + SYNTHESIS (PASS 2)",
    critic_synthesis_response,
)
scribe_log(
    blackboard,
    "CRITIC",
    "synthesis_complete",
    "Pass 2 final evaluation and synthesis complete. Ready for Director review.",
)
validate_stage(blackboard, "critic")

# Update critic_output to refined version for downstream cells
critic_output = critic_synthesis_response

print("── CRITIC FINAL EVALUATION + SYNTHESIS ─────────────────────")
print(critic_synthesis_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Critic pass 2 complete — Creative Team submission ready")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 4e — CRITIC: FINAL EVALUATION + SYNTHESIS (PASS 2)
════════════════════════════════════════════════════════════
── CRITIC FINAL EVALUATION + SYNTHESIS ─────────────────────
IDEA_ID: A1  
ASSESSMENT: The refinement holds well. It keeps the concept operationally realistic for a farmers market while adding a clear brand signature, which makes the single-item approach feel intentional rather than merely minimal.  
REMAINING CONCERN: The signature ritual may still be too subtle unless it is highly visible and consistently repeated, otherwise the stand risks reading as competent but generic.

IDEA_ID: A2  
ASSESSMENT: This is stronger than before because it converts an abstract brand feeling into specific customer behavior loops. The weekly-routine logic is now tied to measurable mechanisms that fit how farmers markets actually generate repeat business.  
REMAINING CONCERN: The concept still depends on execution discipline ac

---
## Cell 8 — Stage 5 & 6: Candidate Set + Director Review

Director assembles candidate directions from the ideation cycle,
evaluates them against the brief, and decides whether to present
or send back for another round.

In [13]:
# ── STAGE 5: CANDIDATE DIRECTIONS ────────────────────────────────────────────
# ── STAGE 6: DIRECTOR REVIEW ─────────────────────────────────────────────────

print("═" * 60)
print("STAGE 5 — CANDIDATE DIRECTIONS")
print("STAGE 6 — DIRECTOR REVIEW")
print("═" * 60)

# Read from the latest ideation cycle (cycle 2 — refined)
creator_output = blackboard["ideation_cycles"][-1]["creator_proposals"][-1][
    "raw_response"
]
critic_output = blackboard["ideation_cycles"][-1]["critic_feedback"][-1]["raw_response"]

review_task = f"""You are reviewing the Creative Team\'s final submission before presenting to the PIL.
The team has completed two passes: initial ideation, research integration, and refinement.

BRIEF:
Challenge: {brief['challenge']}
Desired result: {brief['desired_result']}
Constraints: {", ".join(brief['constraints'])}

REFINED CREATOR DIRECTIONS:
{creator_output[:900]}

CRITIC FINAL EVALUATION + SYNTHESIS:
{critic_output[:900]}

Your task:
1. Select 3 directions to present — from the refined set plus the Critic\'s synthesis (S1).
   Include S1 if it is stronger than any of the three refined directions.
2. Evaluate the candidate set.
3. Decide: PRESENT or ITERATE.

Return ONLY a JSON object — no preamble, no markdown fences:

{{
  "candidates": [
    {{"id": "C1", "title": "exact direction name", "type": "evolutionary|evolutionary_variant|conceptual_leap|sacrificial_probe", "source_idea_id": "A1"}},
    {{"id": "C2", "title": "...", "type": "...", "source_idea_id": "A2"}},
    {{"id": "C3", "title": "...", "type": "...", "source_idea_id": "S1"}}
  ],
  "alignment": "one sentence",
  "distinctiveness": "one sentence",
  "balance": "one sentence",
  "clarity": "one sentence",
  "decision": "PRESENT",
  "reason": "one sentence"
}}

Set decision to "ITERATE" with reason if the team\'s work does not meet standard."""

review_response = call_role(
    role="director",
    user_message=review_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=DIRECTOR_MODEL,
    max_tokens=1000,
    brief_doc=read_brief_doc(session_id),
)

# ── Parse Director review — JSON with validation gate ─────────────────────────
review = blackboard["director_review"]

try:
    clean = review_response.strip()
    if clean.startswith("```"):
        clean = clean.split("```")[1]
        if clean.startswith("json"):
            clean = clean[4:]
    parsed = json.loads(clean.strip())

    review["alignment_with_brief"] = parsed.get("alignment", "")
    review["distinctiveness_assessment"] = parsed.get("distinctiveness", "")
    review["balance_assessment"] = parsed.get("balance", "")
    review["clarity_assessment"] = parsed.get("clarity", "")
    decision = parsed.get("decision", "PRESENT")
    review["iteration_required"] = "ITERATE" in decision.upper()

    candidates = parsed.get("candidates", [])
    if not candidates:
        raise ValueError("Director review returned no candidates")

    for c in candidates:
        blackboard["candidate_set"].append(
            {
                "direction_id": c.get(
                    "id", f"D{len(blackboard['candidate_set'])+1:03d}"
                ),
                "internal_type": c.get("type", "evolutionary").lower(),
                "description": c.get("title", ""),
                "reasoning_summary": "",
                "research_influences": [],
                "critic_assessment": {
                    "strengths": [],
                    "concerns": [],
                    "refinement_notes": [],
                },
            }
        )

    print("✓ Director review parsed successfully")
    print(f"  Decision   : {decision}")
    print(f"  Candidates : {len(candidates)}")

except (json.JSONDecodeError, ValueError) as e:
    print(f"✗ Director review parsing failed: {e}")
    print(review_response)
    raise RuntimeError("Director review parsing failed — candidate set empty.")

update_brief_doc(session_id, "DIRECTOR", "DIRECTOR REVIEW", review_response)
scribe_log(
    blackboard,
    "DIRECTOR",
    "review_complete",
    f"Director review complete. Decision: {'ITERATE' if review['iteration_required'] else 'PRESENT'}. "
    f"{len(blackboard['candidate_set'])} candidates selected.",
)

if not review["iteration_required"]:
    validate_stage(blackboard, "candidate_set")

print("── DIRECTOR REVIEW ─────────────────────────────────────────")
print(review_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Director review complete")
print(f"  Candidates selected : {len(blackboard['candidate_set'])}")
print(f"  Iteration required  : {review['iteration_required']}")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 5 — CANDIDATE DIRECTIONS
STAGE 6 — DIRECTOR REVIEW
════════════════════════════════════════════════════════════
✓ Director review parsed successfully
  Decision   : PRESENT
  Candidates : 3
── DIRECTOR REVIEW ─────────────────────────────────────────
{
  "candidates": [
    {
      "id": "C1",
      "title": "The One Sandwich Stand, Finished Clearly",
      "type": "evolutionary",
      "source_idea_id": "A1"
    },
    {
      "id": "C2",
      "title": "The Neighborhood Ritual Brand, Made Measurable",
      "type": "evolutionary_variant",
      "source_idea_id": "A2"
    },
    {
      "id": "C3",
      "title": "The Weekly Favorite, Marked by a Signature Finish",
      "type": "conceptual_leap",
      "source_idea_id": "S1"
    }
  ],
  "alignment": "All three directions directly address the brief by keeping the launch low-cost and market-first, preserving product-led quality, building repeat demand, and staying comp

---
## Cell 8a — Stage 6a: Iteration Loop

Fires only when the Director signals ITERATE.
Re-runs the full Creative Team: Creator → Critic → Researcher.
Then re-runs Director review to produce a new candidate set.
Cell 9 (Presentation) proceeds normally from the updated candidate_set.

If ITERATE is not required, this cell prints a confirmation and passes through.

In [14]:
# ── STAGE 6a: ITERATION LOOP ─────────────────────────────────────────────────

print("═" * 60)
print("STAGE 6a — ITERATION LOOP")
print("═" * 60)

if not review["iteration_required"]:
    print("✓ Director decision: PRESENT — no iteration needed")
    print(f"✓ Candidates in set: {len(blackboard['candidate_set'])}")

else:
    print("⚠ Director decision: ITERATE — re-running full Creative Team")
    print()

    # Clear existing ideation cycle and candidate set for a clean cycle 2
    blackboard["ideation_cycles"] = []
    blackboard["candidate_set"] = []

    # ── CREATOR — CYCLE 2 ─────────────────────────────────────────────────────
    print("── CREATOR: CYCLE 2 ────────────────────────────────────────")

    iteration_context = (
        f"The Director reviewed the first ideation cycle and requested a full re-run.\n\n"
        f"DIRECTOR'S FEEDBACK:\n{review_response}\n\n"
        f"BRIEF:\n"
        f"Challenge: {brief['challenge']}\n"
        f"Context: {brief['context']}\n"
        f"Desired result: {brief['desired_result']}\n"
        f"Constraints: {', '.join(brief['constraints'])}\n\n"
        f"Generate 4 new distinct directions that address the Director's specific concerns above.\n"
        f"Use Verbalized Sampling: include probability scores. Include at least one below 0.15.\n\n"
        f"For each direction:\n"
        f"IDEA_ID: [A1, A2, A3, A4]\n"
        f"TITLE: [short evocative name]\n"
        f"PROBABILITY: [0.0-1.0]\n"
        f"CONCEPT: [2-3 sentences]\n"
        f"RATIONALE: [why this is worth exploring]\n"
        f"EMOTIONAL TERRITORY: [what feeling does this occupy?]"
    )

    creator_response_iter = call_role(
        role="creator",
        user_message=iteration_context,
        context=brief["challenge"],
        blackboard=blackboard,
        model=SESSION_MODEL,
        max_tokens=1200,
        trait_profile=trait_profiles["creator"],
        brief_doc=read_brief_doc(session_id),
    )

    cycle_2 = {
        "cycle_number": 2,
        "creator_proposals": [{"raw_response": creator_response_iter}],
        "critic_feedback": [],
        "synthesized_directions": [],
    }
    blackboard["ideation_cycles"].append(cycle_2)
    update_brief_doc(
        session_id, "CREATOR", "DIRECTIONS — CYCLE 2", creator_response_iter
    )
    scribe_log(
        blackboard,
        "CREATOR",
        "iteration_ideation_complete",
        "Cycle 2 ideation complete following Director iteration request.",
    )
    validate_stage(blackboard, "ideation")

    print(creator_response_iter)
    print()

    # ── CRITIC — CYCLE 2 ──────────────────────────────────────────────────────
    print("── CRITIC: CYCLE 2 ─────────────────────────────────────────")

    critic_task_iter = (
        f"The Creator has proposed a new set of directions following Director feedback.\n\n"
        f"BRIEF CHALLENGE: {brief['challenge']}\n"
        f"DESIRED RESULT: {brief['desired_result']}\n\n"
        f"CREATOR'S NEW DIRECTIONS:\n{creator_response_iter}\n\n"
        f"Evaluate each direction. For each:\n"
        f"IDEA_ID: [matching the Creator's ID]\n"
        f"STRENGTHS: [what genuinely works]\n"
        f"WEAKNESSES: [what is underdeveloped or misaligned]\n"
        f"REFINEMENT: [one concrete suggestion]\n\n"
        f"Then propose ONE synthesis direction:\n"
        f"SYNTHESIS_ID: S1\n"
        f"TITLE: [name]\n"
        f"CONCEPT: [2-3 sentences]\n"
        f"ORIGIN_IDEAS: [which Creator ideas it draws from]\n\n"
        f"Be analytically rigorous."
    )

    critic_response_iter = call_role(
        role="critic",
        user_message=critic_task_iter,
        context=brief["challenge"],
        blackboard=blackboard,
        model=SESSION_MODEL,
        max_tokens=1800,
        trait_profile=trait_profiles["critic"],
        brief_doc=read_brief_doc(session_id),
    )

    blackboard["ideation_cycles"][-1]["critic_feedback"].append(
        {"raw_response": critic_response_iter}
    )
    update_brief_doc(session_id, "CRITIC", "CRITIQUE — CYCLE 2", critic_response_iter)
    scribe_log(
        blackboard,
        "CRITIC",
        "iteration_evaluation_complete",
        "Cycle 2 Critic evaluation complete.",
    )
    validate_stage(blackboard, "critic")

    print(critic_response_iter)
    print()

    # ── RESEARCHER — CYCLE 2 ──────────────────────────────────────────────────
    print("── RESEARCHER: CYCLE 2 ─────────────────────────────────────")

    research_task_iter = (
        f"You are supporting a creative studio session. The Creative Team has completed "
        f"a second ideation cycle.\n\n"
        f"THE BRIEF:\n"
        f"Challenge: {brief['challenge']}\n"
        f"Context: {brief['context']}\n"
        f"Desired result: {brief['desired_result']}\n\n"
        f"CREATOR'S DIRECTIONS (CYCLE 2):\n{creator_response_iter}\n\n"
        f"CRITIC'S EVALUATION (CYCLE 2):\n{critic_response_iter}\n\n"
        f"Your task: source and cite 2-3 external precedents, examples, or factual findings\n"
        f"from outside this problem's domain that could ground or expand what the Creative\n"
        f"Team has produced. Respond to the actual directions above.\n\n"
        f"For each finding:\n"
        f"TOPIC: [what area, domain, or precedent]\n"
        f"SOURCE: [name the specific source, study, institution, or example — be precise]\n"
        f"FINDING: [what it shows]\n"
        f"RELEVANCE: [why this matters for these specific directions]\n\n"
        f"If you have no external knowledge that would genuinely add here, say only:\n"
        f"RESEARCHER: No external contribution warranted for this cycle.\n\n"
        f"You are a sourcing agent, not an analyst. Do not evaluate the Creative Team's work."
    )

    research_response_iter = call_role(
        role="researcher",
        user_message=research_task_iter,
        context=brief["challenge"],
        blackboard=blackboard,
        model=SESSION_MODEL,
        max_tokens=1200,
        trait_profile=trait_profiles["researcher"],
        brief_doc=read_brief_doc(session_id),
    )

    research_entry_iter = {
        "research_id": f"R{len(blackboard['research_trace']) + 1:03d}",
        "initiated_by": "director_iteration",
        "query": "Post-ideation cycle 2 enrichment",
        "sources": ["researcher_knowledge"],
        "summary": research_response_iter,
        "influence_on_ideation": "Cycle 2 post-iteration research",
    }
    blackboard["research_trace"].append(research_entry_iter)
    brief["research_insights"].append(research_response_iter)
    update_brief_doc(
        session_id, "RESEARCHER", "RESEARCH — CYCLE 2", research_response_iter
    )
    scribe_log(
        blackboard,
        "RESEARCHER",
        "iteration_research_delivered",
        f"Cycle 2 research. Entry R{len(blackboard['research_trace']):03d} logged.",
    )

    print(research_response_iter)
    print()

    # ── DIRECTOR REVIEW — CYCLE 2 ─────────────────────────────────────────────
    print("── DIRECTOR REVIEW: CYCLE 2 ────────────────────────────────")

    review_task_iter = (
        f"You are reviewing the studio's second ideation cycle before presenting to the PIL.\n\n"
        f"BRIEF:\n"
        f"Challenge: {brief['challenge']}\n"
        f"Desired result: {brief['desired_result']}\n"
        f"Constraints: {', '.join(brief['constraints'])}\n\n"
        f"CREATOR DIRECTIONS (CYCLE 2):\n{creator_response_iter[:800]}\n\n"
        f"CRITIC EVALUATION (CYCLE 2):\n{critic_response_iter[:800]}\n\n"
        f"Select 3 directions to present. Return ONLY a JSON object — "
        f"no preamble, no markdown fences:\n\n"
        f"{{\n"
        f'  "candidates": [\n'
        f'    {{"id": "C1", "title": "...", "type": "evolutionary|evolutionary_variant|conceptual_leap|sacrificial_probe", "source_idea_id": "A1"}},\n'
        f'    {{"id": "C2", "title": "...", "type": "...", "source_idea_id": "A2"}},\n'
        f'    {{"id": "C3", "title": "...", "type": "...", "source_idea_id": "S1"}}\n'
        f"  ],\n"
        f'  "alignment": "...",\n'
        f'  "distinctiveness": "...",\n'
        f'  "balance": "...",\n'
        f'  "clarity": "...",\n'
        f'  "decision": "PRESENT",\n'
        f'  "reason": "..."\n'
        f"}}"
    )

    review_response_iter = call_role(
        role="director",
        user_message=review_task_iter,
        context=brief["challenge"],
        blackboard=blackboard,
        model=DIRECTOR_MODEL,
        max_tokens=1200,
        brief_doc=read_brief_doc(session_id),
    )

    # Parse cycle 2 Director review
    try:
        clean_iter = review_response_iter.strip()
        if clean_iter.startswith("```"):
            clean_iter = clean_iter.split("```")[1]
            if clean_iter.startswith("json"):
                clean_iter = clean_iter[4:]
        parsed_iter = json.loads(clean_iter.strip())

        review["alignment_with_brief"] = parsed_iter.get("alignment", "")
        review["distinctiveness_assessment"] = parsed_iter.get("distinctiveness", "")
        review["balance_assessment"] = parsed_iter.get("balance", "")
        review["clarity_assessment"] = parsed_iter.get("clarity", "")
        review["iteration_required"] = False  # reset after cycle 2 review

        for c in parsed_iter.get("candidates", []):
            blackboard["candidate_set"].append(
                {
                    "direction_id": c.get(
                        "id", f"D{len(blackboard['candidate_set'])+1:03d}"
                    ),
                    "internal_type": c.get("type", "evolutionary").lower(),
                    "description": c.get("title", ""),
                    "reasoning_summary": "",
                    "research_influences": [],
                    "critic_assessment": {
                        "strengths": [],
                        "concerns": [],
                        "refinement_notes": [],
                    },
                }
            )

        print("✓ Cycle 2 Director review parsed successfully")

    except (json.JSONDecodeError, ValueError) as e:
        print(f"✗ Cycle 2 Director review parsing failed: {e}")
        print(review_response_iter)
        raise RuntimeError(
            "Iteration Director review failed — candidate set still empty.\n"
            "Check the Director prompt or model output above."
        )

    update_brief_doc(
        session_id, "DIRECTOR", "DIRECTOR REVIEW — CYCLE 2", review_response_iter
    )
    scribe_log(
        blackboard,
        "DIRECTOR",
        "iteration_review_complete",
        f"Cycle 2 Director review complete. {len(blackboard['candidate_set'])} candidates selected.",
    )
    validate_stage(blackboard, "candidate_set")

    # Update creator_output and critic_output so downstream cells (9 and 11)
    # read from cycle 2, not cycle 1.
    creator_output = creator_response_iter
    critic_output = critic_response_iter

    print()
    print(f"✓ Iteration complete")
    print(f"✓ Candidates in set : {len(blackboard['candidate_set'])}")
    print(f"✓ Reasoning trace   : {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 6a — ITERATION LOOP
════════════════════════════════════════════════════════════
✓ Director decision: PRESENT — no iteration needed
✓ Candidates in set: 3


---
## Cell 9 — Stage 7: Presentation

Director presents the candidate directions to the PIL.
Each direction includes reasoning. Tone is clear, professional, non-salesy.

In [15]:
# ── STAGE 7: PRESENTATION ────────────────────────────────────────────────────
print("═" * 60)
print("STAGE 7 — PRESENTATION")
print("═" * 60)

# ── Pull all studio outputs the Director needs to write from ──────────────────
creator_output = blackboard["ideation_cycles"][-1]["creator_proposals"][-1][
    "raw_response"
]
critic_output = blackboard["ideation_cycles"][-1]["critic_feedback"][-1]["raw_response"]

# Researcher output — Director needs this to write Research Foundation sections
# that are legible to the PIL, who has never seen the research
researcher_output = (
    blackboard["research_trace"][-1]["summary"]
    if blackboard["research_trace"]
    else "No research conducted this session."
)

# ── Build candidate summary with full identifying info ────────────────────────
candidate_summary = "\n".join(
    [
        f"- ID: {c['direction_id']} | Title: {c.get('title', c['description'][:60])} "
        f"| Type: {c['internal_type']} | Source: {c.get('source_idea_id', 'unknown')}"
        for c in blackboard["candidate_set"]
    ]
)

# ── Presentation task — aligned to director.md v1.9 PRESENTATION section ─────
presentation_task = f"""You are presenting the studio's work to the PIL.
Three directions are ready. This is the deliverable the PIL will read and
make decisions from. It must do full justice to the work the studio produced.

BRIEF CHALLENGE: {brief['challenge']}
DESIRED RESULT: {brief['desired_result']}

CANDIDATE DIRECTIONS:
{candidate_summary}

FULL STUDIO OUTPUT TO DRAW FROM:

CREATOR DIRECTIONS:
{creator_output[:2500]}

CRITIC EVALUATION:
{critic_output[:2500]}

RESEARCHER FINDINGS:
{researcher_output[:2500]}

---

For each direction, use this exact format with these exact section labels:

**[Direction Title]**
A meaningful name that conveys the mechanism — not a marketing label.
The title should tell the PIL what the direction actually does.

**The Core Move**
Eight to twelve sentences. Explain what this direction does, how it works
mechanically, the main tactical details, and what specific challenge it solves
for this person's problem. Develop the idea fully — not just what it is, but
how it operates in practice, what it changes, and why this particular approach
addresses the underlying tension the brief identified. Ground it in what research
and discovery revealed, not abstract principles. The PIL should hear their own
problem in the description and understand the direction well enough to picture it
in action and begin evaluating whether it fits their situation. This is the heart
of the proposal, where the PIL will start to intuitively feel if it works.
If the mechanism, the rationale, and the human context are not all present,
the section is not complete.

**The Innovation**
One sentence only: what the studio found that the PIL could not have arrived at
alone. This is where research insight, creative tension, or an unexpected reframe
gets surfaced. It is the sentence that earns the studio's existence. If it could
apply to any brief, rewrite it.

**What It Asks**
One sentence: what this direction honestly requires — of the PIL, their
stakeholders, or the situation. A direction that seems to cost nothing is either
obvious or incomplete.

**Research Foundation**
Two to four sentences. THE PIL HAS NOT SEEN THE RESEARCH — write as if explaining
it to someone encountering it for the first time. Name the domain, the key
finding, and why it is relevant to this specific direction.
If you reference a named case, city, study, or data point, give one sentence
of context: what happened, what it showed, and why it applies here.
A reference without context is not a research foundation — it is a citation
the PIL cannot evaluate. The substance must be legible without background knowledge.

**Invitation to Go Deeper**
One sentence, specific — not a vague offer. Name the domain of the research or
the precise evaluative concern. Two forms:
- Research anchor: "This direction draws on [named domain] — ask me for the full
  findings if you want to weigh the evidence before deciding."
- Evaluation signal: "The evaluation identified [specific concern] as the main
  risk — ask me for the refinement path if you want to weigh it before deciding."

---

Sequence the directions intentionally — lead with the strongest, end with
the most unexpected or exploratory. If one direction is clearly superior,
express a subtle preference. If they are equally viable, remain neutral.

Close with a single question that invites the PIL's reaction.

Tone: warm, direct, confident. No hype. Never use 'exciting', 'powerful',
'innovative', 'robust', or hollow affirmations."""

presentation_response = call_role(
    role="director",
    user_message=presentation_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=DIRECTOR_MODEL,
    max_tokens=3500,
    brief_doc=read_brief_doc(session_id),
)

# ── Store presentation to blackboard ─────────────────────────────────────────
blackboard["presentation"]["ordered_directions"] = [
    c["direction_id"] for c in blackboard["candidate_set"]
]
blackboard["presentation"]["director_commentary"].append(presentation_response)

scribe_log(
    blackboard,
    "DIRECTOR",
    "presentation_delivered",
    f"Presentation delivered to PIL. {len(blackboard['candidate_set'])} directions presented.",
)

print("── DIRECTOR PRESENTS TO PIL ────────────────────────────────")
print(presentation_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Presentation complete")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 7 — PRESENTATION
════════════════════════════════════════════════════════════
── DIRECTOR PRESENTS TO PIL ────────────────────────────────
I’d lead with the third direction. It is the most complete answer to the problem as you framed it: affordable to launch, memorable enough to avoid generic pizza-shop gravity, and structured to create repeat behavior instead of hoping for it.

**The Weekly Favorite, Marked by a Signature Finish**

**The Core Move**  
This direction launches the business as a one-sandwich farmers-market stand, but treats the stand less like a food booth and more like a weekly appointment people learn to keep. The Chicken Parm remains the hero, which protects the business from premature menu sprawl and keeps startup cost, prep complexity, and equipment needs under control. What changes is the mechanism around it: the sandwich gets one visible finishing move that customers can recognize instantly every w

---
## Cell 10 — Stage 8: User Reaction

PIL responds to the candidate directions.
Director extracts creative signals from the reaction.
These signals determine whether to proceed to synthesis or trigger a second loop.

In [16]:
# ── STAGE 8: USER REACTION ─────────────────────────────────────────────

print("═" * 60)
print("STAGE 8 — USER REACTION")
print("═" * 60)

pil_reaction = input("Your reaction: ").strip()
print()

# Director extracts signals — JSON format prevents markdown parsing failures
signal_task = f"""The PIL responded to the presentation:
"{pil_reaction}"

Extract the creative signals from this reaction.

Return ONLY a JSON object — no preamble, no markdown fences:

{{
  "selected_direction": "<which direction resonated, e.g. C1, C2, C3, or none>",
  "signal_1": "<key preference or instinct revealed>",
  "signal_2": "<what they want more or less of>",
  "signal_3": "<any new insight or direction revealed, or empty string>",
  "second_loop": <true if the reaction reveals new information that would meaningfully change the creative directions, false if the reaction confirms an existing direction>,
  "second_loop_reason": "<why second loop is or isn't needed — one sentence>"
}}

second_loop should be true only if the PIL's reaction reveals something genuinely new
that the Creative Team did not have when they built the directions — a reframe, a
constraint, a signal that substantially changes what the synthesis should be.
It should NOT trigger just because the PIL wants to push a direction further."""

signal_response = call_role(
    role="director",
    user_message=signal_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=DIRECTOR_MODEL,
    max_tokens=800,  # raised from 400 — JSON response needs room
    brief_doc=read_brief_doc(session_id),
)

# Parse JSON response — with fallback if Director returns narrative instead
try:
    clean = signal_response.strip()
    if clean.startswith("```"):
        clean = clean.split("```")[1]
        if clean.startswith("json"):
            clean = clean[4:]
    signal_data = json.loads(clean.strip())
    second_loop_needed = bool(signal_data.get("second_loop", False))
    second_loop_reason = signal_data.get("second_loop_reason", "")
    signals_list = [
        signal_data.get("signal_1", ""),
        signal_data.get("signal_2", ""),
        signal_data.get("signal_3", ""),
    ]
    signals_list = [s for s in signals_list if s.strip()]
    selected_dir = signal_data.get("selected_direction", "")
except (json.JSONDecodeError, ValueError, AttributeError):
    # Fallback: Director returned narrative — use as-is, no second loop
    second_loop_needed = False
    second_loop_reason = "JSON parse failed — narrative response stored"
    signals_list = []
    selected_dir = ""
    print("  ⚠ Signal extraction returned non-JSON — stored as narrative")

# Store user response
user_resp = blackboard["user_response"]
user_resp["feedback_summary"] = pil_reaction
user_resp["signals_extracted"].append(signal_response)
if selected_dir:
    user_resp["selected_direction"] = selected_dir

# Trigger second loop if needed
blackboard["second_exploration"]["triggered"] = second_loop_needed
if second_loop_needed:
    blackboard["second_exploration"]["reason"] = second_loop_reason

# Update Studio Brief Document
update_brief_doc(
    session_id,
    "DIRECTOR",
    "PIL SIGNALS",
    f"**PIL reaction:** {pil_reaction}\n\n"
    f"**Selected:** {selected_dir}\n\n"
    f"**Signals extracted:**\n{signal_response}\n\n"
    f"**Second loop:** {second_loop_needed}"
    + (f" — {second_loop_reason}" if second_loop_reason else ""),
)

scribe_log(
    blackboard,
    "DIRECTOR",
    "signals_extracted",
    f"PIL reaction processed. Second loop: {second_loop_needed}. "
    f"Key signal: '{pil_reaction[:60]}...'",
)

print("── CREATIVE SIGNALS EXTRACTED ──────────────────────────────────")
print(signal_response)
print("────────────────────────────────────────────────────────────────")
print(f"\n✓ Signals extracted")
print(f"  Selected direction  : {selected_dir or 'not parsed'}")
print(f"  Second loop         : {second_loop_needed}")
if second_loop_needed:
    print(f"  Reason              : {second_loop_reason}")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 8 — USER REACTION
════════════════════════════════════════════════════════════

── CREATIVE SIGNALS EXTRACTED ──────────────────────────────────
{
  "selected_direction": "C3",
  "signal_1": "They strongly prefer the integrated simple launch path that keeps the business tightly focused and operationally manageable.",
  "signal_2": "They want more simplicity and restraint, and less premature expansion into separate product lines.",
  "signal_3": "They were independently considering sauce as a separate business, which reveals an underlying pull toward a broader brand asset beyond the sandwich, even though their current instinct is to avoid complexity at launch.",
  "second_loop": false,
  "second_loop_reason": "This reaction mainly confirms the strongest existing direction and surfaces a future expansion consideration, but it does not materially change the launch strategy the directions were built around."
}
─────────────

---
## Cell 10b — Stage 8b: Depth Extraction

Director uses one extraction technique from the toolkit to probe deeper
before committing to final synthesis.
Director asks one focused question, PIL responds via live input.

In [17]:
# ── STAGE 8b: DEPTH EXTRACTION ───────────────────────────────────────────────
#
# Director probes one level deeper before committing to synthesis.
# Minimum signal threshold: if the PIL's response is noncommittal
# (≤10 words or clearly thin), the Director probes once more with
# a different question. Maximum one re-probe — if the second response
# is also thin, log it and move on.

print("═" * 60)
print("STAGE 8b — DEPTH EXTRACTION")
print("═" * 60)

depth_task = (
    f"The PIL has reacted to the presentation. Before producing the final synthesis,\n"
    f"probe one level deeper using a single extraction technique from your toolkit.\n\n"
    f"PIL REACTION:\n{pil_reaction}\n\n"
    f"SIGNALS EXTRACTED SO FAR:\n{signal_response}\n\n"
    f"Choose ONE technique that would surface the most useful additional signal right now.\n"
    f"Ask a single focused question — not multiple questions.\n"
    f"Keep it conversational. Do not explain why you're asking. Just ask."
)

depth_question = call_role(
    role="director",
    user_message=depth_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=DIRECTOR_MODEL,
    max_tokens=150,
    brief_doc=read_brief_doc(session_id),
)

print("── DIRECTOR DEPTH QUESTION ─────────────────────────────────")
print(depth_question)
print()
depth_response = input("> ").strip()

# -- Minimum signal threshold -------------------------------------------------
# A response of ≤10 words or clearly noncommittal adds little value.
# One re-probe with a different technique before logging.

_thin_signal = len(depth_response.split()) <= 10

if _thin_signal:
    print()
    print("  (Director probing further...)")
    print()

    reprobe_task = (
        f"The PIL gave a brief or noncommittal response to your depth question.\n\n"
        f"YOUR QUESTION: {depth_question}\n"
        f"PIL RESPONSE: {depth_response}\n\n"
        f"Try a different extraction technique — one that approaches from a different "
        f"angle and is more likely to open something up. Do not repeat the same question "
        f"or the same technique. One question only. Keep it light and genuinely curious."
    )

    reprobe_question = call_role(
        role="director",
        user_message=reprobe_task,
        context=brief["challenge"],
        blackboard=blackboard,
        model=DIRECTOR_MODEL,
        max_tokens=150,
        brief_doc=read_brief_doc(session_id),
    )

    print("── DIRECTOR FOLLOW-UP ──────────────────────────────────────")
    print(reprobe_question)
    print()
    reprobe_response = input("> ").strip()

    # Use whichever response has more substance
    if len(reprobe_response.split()) > len(depth_response.split()):
        depth_response = reprobe_response
        print(f"  (Using follow-up response — more signal)")
    else:
        print(f"  (Using original response — follow-up did not add signal)")

# -- Store depth signal -------------------------------------------------------
blackboard["user_response"]["signals_extracted"].append(
    f"DEPTH SIGNAL (Stage 8b): {depth_response}"
)
update_brief_doc(
    session_id,
    "DIRECTOR",
    "DEPTH SIGNAL",
    f"**Director question:** {depth_question}\n\n"
    f"**PIL response:** {depth_response}"
    + (
        f"\n\n**Note:** Thin signal detected — re-probe attempted."
        if _thin_signal
        else ""
    ),
)
scribe_log(
    blackboard,
    "DIRECTOR",
    "depth_extraction_complete",
    f"Stage 8b depth extraction complete. Signal: '{depth_response[:80]}'",
)

print()
print("✓ Depth signal captured")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 8b — DEPTH EXTRACTION
════════════════════════════════════════════════════════════
── DIRECTOR DEPTH QUESTION ─────────────────────────────────
Got it — if the sauce never became a separate product, and only ever lived inside the sandwich experience, would that feel like a relief or like you'd be leaving something important on the table?


  (Director probing further...)

── DIRECTOR FOLLOW-UP ──────────────────────────────────────
Interesting. Fast gut check: imagine it’s six months in and the stand is working — what’s the part you’d be secretly most proud of: that people line up for the sandwich, that regulars know you by name, or that people start asking if they can take a piece of it home?

  (Using follow-up response — more signal)

✓ Depth signal captured
✓ Reasoning trace: 38 entries



---
## Cell 10c — Stage 8c: Second Loop
 
Fires only when the Director's signal extraction identified new information
that warrants a fresh Creative Team run.
 
If second_exploration.triggered is True: re-runs Creator → Critic → Researcher
with the PIL's reaction signals as additional brief context.
Updated directions replace the original candidate set for synthesis.
 
If False: passes through immediately. No API calls made.


In [18]:
# ── STAGE 8c: FOCUSED REFINEMENT ─────────────────────────────────────────────
#
# Fires only when second_exploration.triggered is True.
#
# Design: the second loop is a funnel, not a new exploration.
# The Director restates the brief based on PIL feedback, produces
# 1-2 targeted directions (Creator only, no Critic), conditionally
# calls the Researcher, selects 2-3 final candidates based on the
# rejection signal type, presents them to the PIL, gets a reaction,
# and logs the final signal before synthesis.
#
# If not triggered: passes through immediately.

print("═" * 60)
print("STAGE 8c — FOCUSED REFINEMENT")
print("═" * 60)

if not blackboard["second_exploration"]["triggered"]:
    print("✓ Second loop not required — proceeding to synthesis")
    print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

else:
    print("⚠ Second loop triggered — focused refinement pass\n")
    print(f"  Reason: {blackboard['second_exploration']['reason']}\n")

    all_signals = "\n\n".join(blackboard["user_response"]["signals_extracted"])

    # ── Step 1: Director restatement ─────────────────────────────────────────
    # The Director reads the PIL reaction and existing work, then:
    # (a) writes a focused brief addition naming what needs to change
    # (b) classifies the rejection signal type
    # (c) decides whether new research is needed

    print("── DIRECTOR: RESTATEMENT ────────────────────────────────────")

    # Summarize first-loop candidates for Director context
    first_loop_summary = "\n".join(
        f"- {c['direction_id']}: {c['description']}"
        for c in blackboard["candidate_set"]
    )

    restatement_task = (
        f"You are the Director. The PIL has reacted to the first set of directions "
        f"and a second creative pass has been triggered.\n\n"
        f"ORIGINAL BRIEF CHALLENGE:\n{brief['challenge']}\n\n"
        f"FIRST LOOP DIRECTIONS:\n{first_loop_summary}\n\n"
        f"PIL REACTION AND SIGNALS:\n{all_signals}\n\n"
        f"Do three things and return ONLY a JSON object — no preamble:\n\n"
        f"{{\n"
        f'  "rejection_type": "flat | partial | reframe",\n'
        f'  "surviving_direction": "direction_id of strongest first-loop direction to carry forward, or null if flat rejection",\n'
        f'  "restated_brief": "2-3 sentences: what specifically needs to change or be added based on PIL reaction",\n'
        f'  "research_needed": true or false,\n'
        f'  "research_question": "one specific question if research_needed is true, else null"\n'
        f"}}\n\n"
        f"rejection_type definitions:\n"
        f"- flat: PIL rejected all directions; nothing carries forward\n"
        f"- partial: PIL showed energy toward one direction but it didn't land fully; "
        f"that direction may carry forward alongside new work\n"
        f"- reframe: PIL didn't dislike the directions but identified a missing dimension; "
        f"new directions address what was missing\n\n"
        f"surviving_direction: only relevant for partial — the direction the PIL showed "
        f"most energy toward. Null for flat and reframe.\n\n"
        f"research_needed: true only if the restated brief requires genuinely new external "
        f"knowledge not already present in the session. The Researcher has already "
        f"contributed substantially — default to false unless the gap is clear."
    )

    restatement_response = call_role(
        role="director",
        user_message=restatement_task,
        context=brief["challenge"],
        blackboard=blackboard,
        model=DIRECTOR_MODEL,
        max_tokens=400,
        brief_doc=read_brief_doc(session_id),
    )

    # Parse Director restatement
    try:
        clean_r = restatement_response.strip()
        if clean_r.startswith("```"):
            clean_r = clean_r.split("```")[1]
            if clean_r.startswith("json"):
                clean_r = clean_r[4:]
        restatement = json.loads(clean_r.strip())
        rejection_type = restatement.get("rejection_type", "reframe")
        surviving_direction = restatement.get("surviving_direction")
        restated_brief = restatement.get("restated_brief", "")
        research_needed = bool(restatement.get("research_needed", False))
        research_question = restatement.get("research_question")
    except (json.JSONDecodeError, ValueError):
        rejection_type = "reframe"
        surviving_direction = None
        restated_brief = blackboard["second_exploration"]["reason"]
        research_needed = False
        research_question = None
        print("  ⚠ Restatement parse failed — using trigger reason as restated brief")

    print(f"  Rejection type     : {rejection_type}")
    print(f"  Surviving direction: {surviving_direction or 'none'}")
    print(f"  Research needed    : {research_needed}")
    print(f"  Restated brief     : {restated_brief}")
    print()

    update_brief_doc(
        session_id,
        "DIRECTOR",
        "FOCUSED REFINEMENT — RESTATEMENT",
        f"**Rejection type:** {rejection_type}\n\n"
        f"**PIL signal:** {all_signals}\n\n"
        f"**Restated brief:** {restated_brief}\n\n"
        f"**Surviving direction:** {surviving_direction or 'none'}\n\n"
        f"**Research needed:** {research_needed}",
    )
    scribe_log(
        blackboard,
        "DIRECTOR",
        "restatement_complete",
        f"Restatement complete. Rejection: {rejection_type}. "
        f"Research needed: {research_needed}.",
    )

    # ── Step 2: Conditional Researcher call ──────────────────────────────────
    focused_research = ""

    if research_needed and research_question:
        print("── RESEARCHER: FOCUSED RESEARCH ─────────────────────────────")

        focused_research_task = (
            f"You are supporting a creative studio session in a focused refinement pass.\n\n"
            f"BRIEF CHALLENGE: {brief['challenge']}\n\n"
            f"SPECIFIC RESEARCH QUESTION:\n{research_question}\n\n"
            f"Answer this one question with a named source, finding, and direct "
            f"relevance to the restated brief. Be concise — this is a targeted "
            f"response, not a full research pass. The session already has substantial "
            f"research context. Address only this specific gap."
        )

        focused_research = call_role(
            role="researcher",
            user_message=focused_research_task,
            context=brief["challenge"],
            blackboard=blackboard,
            model=SESSION_MODEL,
            max_tokens=1200,
            trait_profile=trait_profiles["researcher"],
            brief_doc=read_brief_doc(session_id),
        )

        blackboard["research_trace"].append(
            {
                "research_id": f"R{len(blackboard['research_trace']) + 1:03d}",
                "initiated_by": "director_focused_refinement",
                "query": research_question,
                "sources": ["researcher_knowledge"],
                "summary": focused_research,
                "influence_on_ideation": "Focused refinement research delivered",
            }
        )
        update_brief_doc(
            session_id, "RESEARCHER", "RESEARCH — FOCUSED REFINEMENT", focused_research
        )
        scribe_log(
            blackboard,
            "RESEARCHER",
            "second_loop_research_delivered",
            f"Focused refinement research delivered. "
            f"Entry R{len(blackboard['research_trace']):03d} logged.",
        )
        print(focused_research)
        print()
    else:
        print("── RESEARCHER: SKIPPED ──────────────────────────────────────")
        print("  Existing research sufficient — no new call needed.")
        print()

    # ── Step 3: Creator focused pass ─────────────────────────────────────────
    print("── CREATOR: FOCUSED PASS ────────────────────────────────────")

    # Build full context for Creator — includes existing research
    existing_research_summary = "\n\n".join(
        r["summary"][:500] for r in blackboard["research_trace"]
    )

    creator_task_2 = (
        f"You are the Creator. The PIL has reacted to the first set of directions "
        f"and a focused refinement is needed.\n\n"
        f"ORIGINAL BRIEF: {brief['challenge']}\n\n"
        f"WHAT NEEDS TO CHANGE (from Director restatement):\n{restated_brief}\n\n"
        f"PIL REACTION:\n{all_signals}\n\n"
        f"EXISTING RESEARCH CONTEXT (summarized):\n{existing_research_summary}\n\n"
        + (f"NEW RESEARCH:\n{focused_research}\n\n" if focused_research else "")
        + f"Generate 1-2 focused directions (not three) that directly address "
        f"what the PIL identified as missing. Use Verbalized Sampling — assign "
        f"your bands after Step 0.\n\n"
        f"Step 0: Name what the first loop missed in one sentence. "
        f"Then generate from inside that gap.\n\n"
        f"For each direction:\n"
        f"IDEA_ID: [B1, B2]\n"
        f"BAND: [HIGH / MID / LOW]\n"
        f"PROBABILITY: [0.0–1.0]\n"
        f"TITLE: [memorable name]\n"
        f"CONCEPT: [2-3 sentences — what it is]\n"
        f"RATIONALE: [specifically how this addresses what the PIL said was missing]\n"
        f"EMOTIONAL TERRITORY: [how it feels]\n\n"
        f"These must address the gap the PIL identified — not variations on "
        f"what was already presented."
    )

    creator_output_2 = call_role(
        role="creator",
        user_message=creator_task_2,
        context=brief["challenge"],
        blackboard=blackboard,
        model=SESSION_MODEL,
        max_tokens=1800,
        trait_profile=trait_profiles["creator"],
        brief_doc=read_brief_doc(session_id),
    )

    blackboard["second_exploration"]["additional_cycles"].append(
        {
            "cycle_number": len(blackboard["ideation_cycles"]) + 1,
            "creator_proposals": [{"raw_response": creator_output_2}],
            "critic_feedback": [],
            "synthesized_directions": [],
        }
    )
    update_brief_doc(
        session_id, "CREATOR", "FOCUSED REFINEMENT DIRECTIONS", creator_output_2
    )
    scribe_log(
        blackboard,
        "CREATOR",
        "second_loop_ideation_complete",
        "Focused refinement: new directions generated targeting PIL gap.",
    )
    print(creator_output_2)
    print()

    # ── Step 4: Director selection ────────────────────────────────────────────
    # Build final candidate set based on rejection type:
    # flat    → new directions only (1-2)
    # partial → strongest survivor + new directions (max 3)
    # reframe → new directions only, framed as addressing the gap

    print("── DIRECTOR: SELECTION ──────────────────────────────────────")

    # Parse new B-series directions
    b_ids = re.findall(r"IDEA_ID:\s*(B\d+)", creator_output_2)
    b_titles = re.findall(r"TITLE:\s*(.+)", creator_output_2)

    new_candidates = []
    for i, bid in enumerate(b_ids[:2]):
        new_candidates.append(
            {
                "direction_id": f"C{i+1}",
                "internal_type": "second_loop",
                "description": b_titles[i].strip() if i < len(b_titles) else bid,
                "reasoning_summary": "",
                "research_influences": [],
                "critic_assessment": {
                    "strengths": [],
                    "concerns": [],
                    "refinement_notes": [],
                },
            }
        )

    # Build final candidate set
    if rejection_type == "partial" and surviving_direction:
        # Find surviving direction from first loop
        survivor = next(
            (
                c
                for c in blackboard["candidate_set"]
                if c["direction_id"] == surviving_direction
            ),
            None,
        )
        if survivor:
            # New directions first, survivor appended
            final_candidates = new_candidates + [survivor]
        else:
            final_candidates = new_candidates
    else:
        # flat or reframe: new directions only
        final_candidates = new_candidates

    # Renumber cleanly
    for i, c in enumerate(final_candidates):
        c["direction_id"] = f"C{i+1}"

    blackboard["candidate_set"] = final_candidates[:3]  # hard cap at 3

    # Update creator_output for Cell 11 synthesis
    creator_output = creator_output_2

    scribe_log(
        blackboard,
        "DIRECTOR",
        "second_loop_complete",
        f"Focused refinement complete. {len(blackboard['candidate_set'])} "
        f"candidates selected ({rejection_type} rejection). Presenting to PIL.",
    )

    print(f"  Rejection type : {rejection_type}")
    print(f"  Final set      : {len(blackboard['candidate_set'])} directions")
    for c in blackboard["candidate_set"]:
        print(
            f"    {c['direction_id']} [{c['internal_type']}]: {c['description'][:60]}"
        )
    print()

    # ── Step 5: Director presents refined directions to PIL ───────────────────
    print("── DIRECTOR: REFINED PRESENTATION ──────────────────────────")

    refined_set_summary = "\n".join(
        f"- {c['direction_id']}: {c['description']}"
        for c in blackboard["candidate_set"]
    )

    # Frame the presentation based on rejection type
    if rejection_type == "flat":
        framing_note = (
            "The PIL rejected all first directions. Present these as a completely "
            "fresh approach that addresses what they said was missing. Do not "
            "reference the first directions at all."
        )
    elif rejection_type == "partial":
        framing_note = (
            f"The PIL showed some energy toward {surviving_direction or 'one direction'} "
            f"but it didn't fully land. Present the new directions as addressing the gap, "
            f"and carry the strongest first-loop direction forward as still relevant."
        )
    else:  # reframe
        framing_note = (
            "The PIL identified a missing dimension. Present these as directly "
            "addressing that dimension — not as replacements for everything, "
            "but as what was missing from the first pass."
        )

    presentation_task_2 = (
        f"Present these refined directions to the PIL.\n\n"
        f"BRIEF: {brief['challenge']}\n\n"
        f"WHAT PIL SAID WAS MISSING: {restated_brief}\n\n"
        f"DIRECTIONS TO PRESENT:\n{refined_set_summary}\n\n"
        f"FULL CREATOR OUTPUT FOR REFERENCE:\n{creator_output_2}\n\n"
        f"FRAMING GUIDANCE: {framing_note}\n\n"
        f"Present each direction clearly with 2-3 sentences of substance. "
        f"Include one transparency hook per direction — a brief sentence "
        f"inviting the PIL to ask about research or evaluation if they want "
        f"to go deeper. "
        f"Close by asking which direction resonates, or whether they want to "
        f"proceed directly to synthesis."
    )

    refined_presentation = call_role(
        role="director",
        user_message=presentation_task_2,
        context=brief["challenge"],
        blackboard=blackboard,
        model=DIRECTOR_MODEL,
        max_tokens=600,
        brief_doc=read_brief_doc(session_id),
    )

    print(refined_presentation)
    print()

    # ── Step 6: PIL reacts to refined directions ──────────────────────────────
    pil_refined_reaction = input("Your reaction: ").strip()

    # ── Step 7: Director reads reaction and logs ──────────────────────────────
    refined_signal_task = (
        f"The PIL has responded to the refined directions:\n\n"
        f"REFINED DIRECTIONS PRESENTED:\n{refined_set_summary}\n\n"
        f"PIL RESPONSE:\n{pil_refined_reaction}\n\n"
        f"Read this response. In one or two sentences:\n"
        f"1. Acknowledge what they said naturally (do not be sycophantic)\n"
        f"2. Confirm which direction or combination you will synthesize toward\n\n"
        f"Keep it brief and decisive. The PIL has given their input — "
        f"now confirm the direction and move toward synthesis."
    )

    refined_signal_response = call_role(
        role="director",
        user_message=refined_signal_task,
        context=brief["challenge"],
        blackboard=blackboard,
        model=DIRECTOR_MODEL,
        max_tokens=150,
        brief_doc=read_brief_doc(session_id),
    )

    print()
    print(refined_signal_response)
    print()

    # Store refined reaction signals
    blackboard["user_response"]["signals_extracted"].append(
        f"REFINED PASS SIGNAL: {pil_refined_reaction}"
    )
    blackboard["second_exploration"][
        "reason"
    ] += f"\n\nRefined pass PIL reaction: {pil_refined_reaction}"

    update_brief_doc(
        session_id,
        "DIRECTOR",
        "REFINED PRESENTATION + PIL REACTION",
        f"**Director presented:**\n{refined_presentation}\n\n"
        f"**PIL responded:**\n{pil_refined_reaction}\n\n"
        f"**Director acknowledged:**\n{refined_signal_response}",
    )
    scribe_log(
        blackboard,
        "DIRECTOR",
        "refined_presentation_complete",
        f"Refined directions presented and PIL reacted. "
        f"Signal: '{pil_refined_reaction[:60]}'",
    )

    print("── FOCUSED REFINEMENT COMPLETE ──────────────────────────────")
    print(f"  Final candidates  : {len(blackboard['candidate_set'])}")
    for c in blackboard["candidate_set"]:
        print(
            f"    {c['direction_id']} [{c['internal_type']}]: {c['description'][:60]}"
        )
    print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 8c — FOCUSED REFINEMENT
════════════════════════════════════════════════════════════
✓ Second loop not required — proceeding to synthesis
✓ Reasoning trace: 38 entries


---
## Cell 11 — Stage 9: Final Synthesis

Director produces the final synthesis — refining the chosen direction
and incorporating signals from the PIL's reaction.
This is the studio's best work.

In [19]:
# ── STAGE 9: FINAL SYNTHESIS ─────────────────────────────────────────────────
# v3.2 patch: research-anchored action plan, max_tokens raised to 1400

print("═" * 60)
print("STAGE 9 — FINAL SYNTHESIS")
print("═" * 60)

all_signals = "\n\n".join(blackboard["user_response"]["signals_extracted"])
signals = all_signals if all_signals else ""

# Pull research summary from research_trace for synthesis integration
research_summary = ""
if blackboard.get("research_trace"):
    research_summary = "\n\n".join(
        f"FINDING: {entry.get('summary', '')[:600]}"
        for entry in blackboard["research_trace"]
        if entry.get("summary")
    )

synthesis_task = f"""Produce the final synthesis for the PIL.

BRIEF: {brief['challenge']}
DESIRED RESULT: {brief['desired_result']}

PIL REACTION: {pil_reaction}

SIGNALS EXTRACTED:
{signals}

STUDIO DIRECTIONS EXPLORED:
{creator_output[:400]}

RESEARCH PRODUCED BY THE STUDIO:
{research_summary[:2000]}

Synthesize the most compelling direction — refined by the PIL's signals.
This may combine elements from multiple directions.

Name this document appropriately for the type of problem solved. Use terms
such as: Action Plan, Creative Direction, Business Launch Plan, Brand Strategy,
Product Roadmap, or similar. Do not use "Final Synthesis" under any circumstances.

Begin with a single sentence introduction — warm, direct, reflective of genuine
authorship by the full creative team, informed by everything the PIL shared.

Structure:

[DOCUMENT NAME]: [title]

[intro sentence]

CORE CONCEPT:
3-4 sentences. The essential idea, grounded in what discovery and research revealed.

WHY THIS WORKS:
2-3 sentences. Name the specific insight from the research or evaluation that makes
this direction sound — not just logical, but evidenced. If the research found a
named precedent, a documented outcome, or a specific mechanism that validates this
direction, name it here in plain language.

ACTION PLAN:
Organize as 2-3 phases, not a flat list. Each phase has:
- A short phase name and approximate time horizon
- 1-2 specific actions
- One sentence connecting the action to what the research found — why this
  specific approach is the right move, drawn from the studio's findings

The research connection should be concrete and specific, not generic. If the
research named a methodology, a case study, a mechanism of failure, or a
documented outcome that directly informs an action step, use it. This is what
makes the plan feel earned rather than invented.

Do not fabricate contact details, URLs, or current organizational information —
these require live verification. Where specific people or organizations are named,
note them as starting points to research rather than confirmed contacts.

Close with a single sentence inviting the PIL's reaction. Open and genuine.

Tone: confident, clear, earned. The PIL should leave feeling that the studio
understood their problem and built something real for them."""

synthesis_response = call_role(
    role="director",
    user_message=synthesis_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=DIRECTOR_MODEL,
    max_tokens=1600,
    brief_doc=read_brief_doc(session_id),
)

# Store final synthesis
synthesis = blackboard["final_synthesis"]
synthesis["summary"] = synthesis_response
if blackboard["candidate_set"]:
    synthesis["final_direction"] = blackboard["candidate_set"][-1]["direction_id"]
synthesis["refinements"].append(f"Refined from PIL signal: '{pil_reaction[:100]}'")

scribe_log(
    blackboard,
    "DIRECTOR",
    "synthesis_complete",
    "Final synthesis delivered to PIL. Session complete.",
)

print("── FINAL SYNTHESIS ─────────────────────────────────────────")
print(synthesis_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Final synthesis complete")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 9 — FINAL SYNTHESIS
════════════════════════════════════════════════════════════
── FINAL SYNTHESIS ─────────────────────────────────────────
Business Launch Plan: The Weekly Favorite

The strongest version of this is a simple market-first sandwich business built around one Chicken Parm people learn to line up for because it is reliably there, unmistakably yours, and good enough to become part of their weekly routine.

CORE CONCEPT:
Start with one hero product and make the business legible through repetition: same sandwich, same market rhythm, same visible signature finish, same reason to come back. The launch should be built for farmers-market reality, not restaurant fantasy — prep from an approved kitchen or commissary, tight service flow, and a presentation ritual that makes the sandwich feel memorable without adding operational drag. The brand is not “Italian food” or “red sauce”; it is the sandwich people know they

---
## Cell 11b — Stage 9b: Research  Appendix

The studio's full research output, surfaced after synthesis for PIL review and follow-up.

In [20]:
# ── STAGE 9b: RESEARCH APPENDIX ──────────────────────────────────────────────
# Prints full research trace as a readable appendix after final synthesis.
# All findings surfaced for PIL review. Future: downloadable document.

print("═" * 60)
print("RESEARCH APPENDIX")
print("Studio findings that informed this session")
print("═" * 60)

research_trace = blackboard.get("research_trace", [])

if not research_trace:
    print("No research findings recorded in this session.")
else:
    print(f"This session produced {len(research_trace)} research contribution(s).\n")
    print(
        "These findings informed the directions and synthesis above.\n"
        "They are presented here in full for your reference.\n"
    )
    print("─" * 60)

    for i, entry in enumerate(research_trace):
        summary = entry.get("summary", "")
        if not summary:
            continue

        print(f"\nRESEARCH CONTRIBUTION {i + 1}")
        print(f"Initiated by: {entry.get('initiated_by', 'creative_team')}")
        print(f"Query: {entry.get('query', '')}")
        print("─" * 40)
        print(summary)
        print()

    print("─" * 60)
    print(
        "\nNote: Research findings are drawn from the studio's training knowledge.\n"
        "Named sources, individuals, and institutions should be independently\n"
        "verified before being used in outreach or decision-making.\n"
        "Links and current contact information require live web research."
    )

print(f"\n✓ Research appendix complete")
print(f"✓ {len(research_trace)} research entry/entries surfaced")

════════════════════════════════════════════════════════════
RESEARCH APPENDIX
Studio findings that informed this session
════════════════════════════════════════════════════════════
This session produced 1 research contribution(s).

These findings informed the directions and synthesis above.
They are presented here in full for your reference.

────────────────────────────────────────────────────────────

RESEARCH CONTRIBUTION 1
Initiated by: creative_team
Query: Targeted requests + autonomous contribution post-ideation
────────────────────────────────────────
TOPIC: Farmers market vending rules for hot sandwich service  
SOURCE: U.S. Food and Drug Administration, *Model Food Code* (2022); local county/city health department food-vending permit rules; farmers market manager vendor guidelines  
FINDING: In the U.S., hot sandwich service at farmers markets is usually governed by a combination of temporary food establishment rules, commissary or approved kitchen requirements, handwashing 

---
## Cell 12 — Session Record: Blackboard Inspection + Save

Complete session record. Full reasoning trace visible.
Session saved to JSON — raw material for Phase 3 visualization.

In [21]:
# ── SESSION RECORD ────────────────────────────────────────────────────────────

print("═" * 60)
print("SESSION RECORD")
print("═" * 60)
print(f"Session ID      : {blackboard['session_metadata']['session_id']}")
print(f"Prompt          : {blackboard['user_prompt']}")
print(f"Model           : {SESSION_MODEL}")
print()
print(f"Ideation cycles : {len(blackboard['ideation_cycles'])}")
print(f"Research entries: {len(blackboard['research_trace'])}")
print(f"Candidate dirs  : {len(blackboard['candidate_set'])}")
print(f"Reasoning steps : {len(blackboard['reasoning_trace'])}")
print()
print("── REASONING TRACE ─────────────────────────────────────────")
for entry in blackboard["reasoning_trace"]:
    print(
        f"  {entry['step']:>2} | {entry['role']:<12} | {entry['action']:<22} | {entry['summary'][:60]}"
    )

print()
print("── CANDIDATE DIRECTIONS ────────────────────────────────────")
for c in blackboard["candidate_set"]:
    print(f"  {c['direction_id']} | {c['internal_type']:<22} | {c['description']}")

# -- Director summary ----------------------------------------------------------
# One-sentence session label written by the Director.
# Stored in metadata for quick identification without opening the full JSON.
# Fix: pass read_brief_doc(session_id) so the Director has full session context.
print()
print("── DIRECTOR SUMMARY ────────────────────────────────────────")

summary_task = (
    "In one sentence, summarize what problem was explored in this session "
    "and what direction the studio landed on. "
    "Write it as a neutral third-party description — not addressed to anyone, "
    "not promotional, not a question. "
    "Be specific to this session: name the domain, the core challenge, and "
    "the nature of the solution direction. Maximum 35 words."
)

try:
    director_summary = call_role(
        role="director",
        user_message=summary_task,
        context=blackboard["user_prompt"],
        blackboard=blackboard,
        model=DIRECTOR_MODEL,
        max_tokens=80,
        brief_doc=read_brief_doc(session_id),
    )
    director_summary = director_summary.strip()
    blackboard["session_metadata"]["director_summary"] = director_summary
    print(f"  {director_summary}")
except Exception as e:
    print(f"  ⚠ Director summary failed: {e}")
    blackboard["session_metadata"]["director_summary"] = ""

# -- Assemble evaluation payload -----------------------------------------------
# Packages problem_prompt, baseline, and final synthesis for evaluator.
assemble_evaluation_payload(blackboard, baseline_response)

# -- Post-session notes --------------------------------------------------------
# Capture observer notes while session is fresh. Stored in JSON metadata.
session_notes = input("\nSession notes (press Enter to skip):\n> ").strip()
if session_notes:
    blackboard["session_metadata"]["notes"] = session_notes

# -- Save full session ---------------------------------------------------------
saved_path = save_session(blackboard)
print()
print(f"✓ Full session saved to: {saved_path}")
print()
print("Phase 1 complete. Full studio workflow executed.")
print()
print("This JSON is Phase 3 input — the semantic trajectory data.")
print("Every reasoning_trace entry is an utterance in conceptual space.")

════════════════════════════════════════════════════════════
SESSION RECORD
════════════════════════════════════════════════════════════
Session ID      : 4d7a1f1f-03c9-4e51-ac1e-057d9f9d84ee
Prompt          : I'm thinking of starting a Chicken Parmesan sandwich business, but I don't know anything about how to do it. My friends tell me that I make a really great one. People ask me to bring them to parties. Can you help me? I was thinking of naming it after my grandmother, Petrina. I don't have a lot of money but I work hard.
Model           : gpt-5.4-mini

Ideation cycles : 2
Research entries: 1
Candidate dirs  : 3
Reasoning steps : 40

── REASONING TRACE ─────────────────────────────────────────
   1 | SYSTEM       | session_start          | Session initiated: 'I'm thinking of starting a Chicken Parme
   2 | DIRECTOR     | api_call               | Responded to: 'A person has just arrived at The Creative Pri
   3 | DIRECTOR     | routing_studio         | Request routed to STUDIO. Enter